# About this Notebook

<div class="alert alert-block alert-warning">This notebook is intended to be run within a workshop that assumes certain infrastructure exists as part of a managed deployment. Certain features will not work outside this environment because specific environment variables will not be defined and infrastructure will not exist. The intention is to produce a standalone version of this content that can be run within your own AWS environment at a later date.</div>
    
In this workshop we're going to build a **knowledge graph** based on an unstructured blog post describing a walking tour of New York City. We will utilize various prompting strategies to get the LLM to extract the entities we desire, relationships between those entities, and output them in a way that is conducive to loading into a graph databases. After we are satisfied with the graph results, we will load the graph into Amazon Neptune, a fully managed graph database engine, and experiment with using an LLM to translate natural language into graph queries for extracting information from our graph. Finally, we will develop an LLM prompt to take the graph query results and form a natural language response. We will use state of the art LLM models from Anthropic and Amazon managed by Amazon Bedrock. 

**From the author:**
This won't be your typical "Everything is neat and scripted and buttoned up" workshop. This is going to be big and messy with an ambiguous data set and asking the LLM to do things it isn't all that good at (e.g., organize locations by the NYC neighborhood they are in). 
What to expect: 
- I'm a firm believer that you learn more by pushing the envelope and failing vs just running through a safe workshop where everything turns out perfectly. 
- My end goal is that you will learn a lot about the importance of the prompt, and prompting techniques that work well and others that do not work so well. 
- Stop here if you are looking for a clean, scripted experience. At least 75% of the participants will fail to generate a graph that will load. One type of edge we ask the LLM to create will cause it to hallucinate on edge ids and generate ones that don't exist (but don't worry, I've included a backup graph that will load so you won't get stuck). I could have taken that edge out of the workshop, but what fun is that? 
- Come prepared to share your own techniques and ideas as well. I want this to be a workshop where I can set you up to develop your own ideas and share them so you can learn from each other.

# Notebook Configuration

This notebook requires at least version 1.37.24 of boto3.  Run this cell to get the current version. New Neptune Workbench instances should have at least this version, but if you are running on your own account and using an older notebook instance, it may not be.

In [1]:
import boto3
boto3.__version__

'1.38.0'

**You only need to run this next cell if the current version is less than 1.37.34.**

In [ ]:
!pip install boto3==1.37.34

Run this cell to capture the notebook configuration data as a variable called config. We use this later in the notebook to capture the region, IAM ARN, and Neptune connection information

In [4]:
%graph_notebook_config --store-to config

{
    "host": "neptunedbcluster-5saebswbnpaa.cluster-cn4m0owg2nrc.us-west-2.neptune.amazonaws.com",
    "neptune_service": "neptune-db",
    "port": 8182,
    "proxy_host": "",
    "proxy_port": 8182,
    "auth_mode": "DEFAULT",
    "load_from_s3_arn": "arn:aws:iam::486973336386:role/cfn-deploy-NeptuneLoadFromS3Role-aLbDTdYtXEp2",
    "ssl": true,
    "ssl_verify": true,
    "aws_region": "us-west-2",
    "sparql": {
        "path": "sparql"
    },
    "gremlin": {
        "connection_protocol": "websockets",
        "traversal_source": "g",
        "username": "",
        "password": "",
        "message_serializer": "GraphSONMessageSerializerV3"
    },
    "neo4j": {
        "username": "neo4j",
        "password": "password",
        "auth": true,
        "database": null
    }
}


This cell will make the notebook span the width of your browser window, which will make it easier to read on wider screens.

In [5]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

<ipython-input-5-ac09909db896>:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


# Raw Text

For our workshop today, we will be using a walking tour of New York City found at https://maycausewanderlust.com/walking-tour-new-york-city/. I've extracted just the text for the walking tour and put it into this variable to prevent unnecessary traffic to their website.

In [6]:
WALKING_TOUR_TEXT = """
The walking tour route I am sharing here takes you from downtown Manhattan, through mid-town to Central Park and the Upper East Side, taking in many NYC icons along the way, including:

Greenwich Village & Washington Square Park
Flatiron Building & Madison Square Park
Empire State Buiding
New York Public Library
Fifth Avenue Shopping
Rockefeller Centre & Top Of The Rock
Plaza Hotel & Pulitzer Fountain
Central Park

There are other routes that I like in Manhattan and other neighbourhoods which are great to explore – but I wanted to start with something that was doable in one day, and which covers several of the major landmarks in New York. I walked this route on a visit to New York with a friend who had never been to the city before – so I have literally road-tested it, haha.

The whole route I am recommending here is just short of 5 miles (8km) and would take approximately an hour and 45 minutes to walk non-stop according to GoogleMaps. However, it will take longer because you’ll be stopping to cross the street, take photos, grab refreshments and go into some of the attractions along the way. Using the photos I took when I walked this route as a reference, it took me around 7 hours to do this route. This included going to the Top of the Rock (but not the Empire State Building) – and I was pretty tired afterwards! That’s why I think you should allow most of the day for this.

When I walk around a new city, I don’t always take a fixed route. If possible, I like to know the general direction I’m headed but to have some freedom to walk down the streets that look most interesting to me.  Therefore, I’d encourage you to take some liberties with the route I recommend and to take detours if you feel like it.

This route does require you to be able to walk for extended distances – and I’ll share some walking tips at the end (as well as a map).

Self-Guided Walking Tour of Manhattan – Step By Step
Start In Greenwich Village
Greenwich Village was the bohemian capital of New York in the 1950s and 60s, the epicentre of art, counter-culture and the LGBT community. These days, the quiet, leafy streets of Greenwich Village are amongst the most expensive places to live in the United States.

Start in your walk in St Luke in the Fields Garden, which is a small church garden on the corner of Hudson and Barrow streets, in the heart of the West Village. It’s a very pretty spot that remains somewhat off the beaten path in New York.

From here, head up Hudson Street then right onto Grove Street, where you might recognise the Friends building – the building that was used for the exterior shots of Monica & Rachel and Joey & Chandler’s apartment in the 1990s TV show. Carry on along Grove Street until you get to Christopher Park, which is home to the Stonewall Inn, the location of landmark riots against police persecution of gay or queer people. There is a monument in the park and some plaques outside the inn if you want to read about the uprising.

If you want to detour from the walking route, here are some other Greenwich Village highlights you could check out:
Joe’s Pizza – a famous and reliably good spot for pizza by the slice
Jefferson Market Library – this historic red brick building with a clocktower was once a courthouse and is now a branch of the New York Public Library
Carrie Bradshaw’s apartment. The exterior of Carrie’s brownstone building was shot at 66 Perry Street (even though she is written as living in the Upper East Side). It’s also just a lovely leafy street to wander along!
Angelika Theatre – a legendary arthouse cinema in the village
Alternative: Start in the East Village
The East Village is another historic neighbourhood in Manhattan – and you could also start your walking tour here, rather than in the West Village, if you prefer.

It was an upscale area of New York, built when the city expanded in the early 1800s. Then in the late 1800s, the East Village’s population swelled with immigrants, at one point being known as ‘Little Germany’.  In the 1950s and 60s, the area absorbed some of the beatnik creativity of neighbouring Greenwich Village. Allen Ginsberg, W. H. Auden, and Norman Mailer all moved to the area in 1951–1953. Like a lot of New York, the area became run down in the 1970s and 80s. Since the early 2000s it has been gentrified – but it does remain somewhat rougher around the edges than Greenwich Village (the West Village in particular).

At the heart of the East Village is Tomkins Square Park, which you can wander around.  But I definitely recommend starting your walking journey with a drink – of water or coffee. Try The Maiden Lane, which is on the corner of Tompkins Square Park at 10th Street and Avenue B, or at B Cup, a couple of blocks north on 13th Street.

From Tompkins Square Park, head west toward Washington Square Park.  The distance is approximately 0.8 miles, so it should take around 17-18 minutes. Along the way, you could check out Veselka, a favourite for Ukrainian cuisine; the attractive houses on Renwick Triangle, and (if you don’t mind a slightly indirect route) the pretty church garden of Grace Church.

Washington Square Park
Whether you start in the West Village, as I suggest, or the East Village, the next stop is Washington Square Park. This is one of the best-known of New York City’s parks and spending time in it is one of the many free things to do in NYC. It features two key landmarks: the ornate marble Washington Square Arch and a large fountain.

Take some time to rest and absorb the energy of the park. Its atmosphere is often vibrant, with street performers and people milling about. In the south-west corner, there are chess boards set up, if you fancy your chances in a match.

Just north of the park is the start of Fifth Avenue and the location of one of the most well-known addresses in Manhattan, the Art Deco skyscraper at One Fifth Avenue.

Union Square Park
From Washington Square Park, walk up University Place towards Union Square Park.  This route takes through the heart of the NYU campus. The distance is 0.4 miles and should take approximately 8-9 minutes. On the way, you’ll pass by a restaurant I like: the Gotham Bar & Grill. Just mentioning it in case you’re hungry for lunch at this point.

Union Square Park also has a chess scene, so you could watch some games here – or play if you fancy your chances! There’s also a statue of George Washington here, and a farmer’s market on Mondays, Wednesdays, Fridays, and Saturdays.

Flatiron Building & Madison Square Park
From Union Square Park, walk up Broadway to Madison Square Park.  The distance is 0.4 miles and should take approximately 8-9 minutes. On the way, if you’re peckish, there’s a great bakery called Levain Bakery a block or so off-course on 18th street. They do the most amazing thick and gooey cookies!

Madison Square Park is surrounded by gorgeous-looking skyscrapers from the early 20th century, including the gold-topped New York Life Building and a clock tower, which was once the Metropolitan Life Insurance Company’s headquarters and is now a hotel.

However,  at the southwest corner of Madison Square Park is one of the first truly iconic buildings you will see on this walking tour of Manhattan: the Flatiron Building. The Flatiron Building is a 20-story steel-framed building that was built in 1902. Officially a New York City Landmark, it is well known and loved for its triangular shape with a narrow-angled corner facing north at the junction of Fifth Avenue and 23rd Street – it really is a marvel!

You’ve been walking a lot by this point, so you may want some refreshments and there’s a great option for that near the Flatiron building at Eataly, which has a great selection of different food counters and eateries in its complex.  My choice was an Italian gelato, which I took outside to eat in Madison Square Park.

Empire State Building
From Madison Square Park, you’ll see your next stop way before you reach it (although I’ve heard locals are rather annoyed by the emergence of a new skyscraper that obscures the view of the ESB from the south)!  Walk up Fifth Avenue towards the impressive and inimitable Empire State Building, which occupies an entire block between 33rd and 34th streets.   The distance is 0.5 miles and should take approximately 9-10 minutes. Along the way, there’s a popular rooftop bar nearby: 230 Fifth Rooftop Bar – in case you’re interested in getting an elevated view at this point.

Along the way, you can snap pictures of this Art Deco architectural masterpiece from various distances – the Empire State Building really does dominate Midtown! It was the tallest building in the world when it was built in 1931 and it held this record until the World Trade Centre towers went up in 1970. 

There are observation decks on the 86th and 102nd floors of the Empire State Building, which give amazing panoramic views of New York City and six states.  Taking the view from the top of the Empire State Building is on many people’s New York bucket list.

However, if you only have an appetite (or time, or budget) for one observation deck, I personally think the Top of the Rock is better. And don’t worry – that is also on this walking tour of Manhattan!

New York Public Library
From the Empire State Building, carry on up Fifth Avenue for 7 blocks until you come to the New York Public Library on the left.  The distance is another 0.5 miles and should take approximately 9-10 minutes. Along the way, be aware that the Morgan Library & Museum is a block over on 37th Street.

By this point, Fifth Avenue is like a man-made canyon: the road is a deep chasm between towering skyscrapers. But the next stop is not a high rise.

The New York Public Library is an NYC institution and has been providing access to books and information for more than 125 years. Even if you’re not in the market for some literature, the building itself is worth a detour from your walk. The Beaux-Arts style building, guarded by two marble lions called Patience and Fortitude, was built in 1911 and is a National Historic Landmark. The whole building is wonderful but the Rose Reading room, with its grand arched windows, chandeliers and rows of lamp-lit tables, is really stunning.

And if you’re a movie fan, like me, you may remember scenes from the New York Public Library in NYC-set movies like Ghostbusters, The Day After Tomorrow and Sex & The City: The Movie.

Once you’ve had your fill of the library itself, do check out the park behind it: Bryant Park is a handsome park with lots of seats and views of the surrounding skyscrapers, including the Art Deco American Radiator Building – and the very top of the Empire State Building.

Rockefeller Centre & Top of the Rock
From the New York Public Library, carry on up Fifth Avenue for another 8 blocks until you come to the Rockefeller Centre.  The distance is another 0.5 miles and should take approximately 9-10 minutes. 

At 42nd Street, look right for a glimpse of the Chrysler Building, which is a couple of blocks east of Fifth Avenue. This is my favourite skyscraper in Manhattan – I just love the elegant Art Deco style and silver finish. If you want to take a detour to see it, there’s a good viewpoint on Lexington Avenue, and you could stop by Grand Central Station at the same time. You’ll also start to see lots more shopping opportunities as you go further north on Fifth Avenue.

At Rockefeller Centre, I highly recommend you go to the observation deck in the Rockefeller Tower. It is called Top of the Rock and is 70 stories high with stunning views of the city – it is one of the best observation decks in New York City.  Looking northwards, you can see Central Park and uptown – It is amazing to see the park from so high up and to see just how huge it is, and to see those ornate towered buildings that line it on both sides. 

However, in my opinion, the best view is south, towards midtown and lower Manhattan.  Standing proud in the middle of this view is the Empire State Building. It really is the cherry on the top of the views of Manhattan. You can’t admire the view of the Empire State Building from the top of the Empire State Building, so that’s why I said earlier the view from the Top of the Rock is the best one.

The Rockefeller Centre is also a good place to grab a bite to eat.  At street level, there’s a restaurant with an outdoor terrace right by the fountains.  It’s not cheap – but it is scenic and the salads are great. And Radio City Hall is just down the street, in case you want to check that out while you’re there.

The Plaza Hotel & The Pulitzer Fountain
From the Rockefeller Centre, carry on up Fifth Avenue for another 9 blocks until the buildings give way to Central Park.  On the corner of Fifth Avenue and 59th Street are the Pulitzer Fountain and the Plaza Hotel.  The distance is another 0.5 miles and should take approximately 9-10 minutes. 

On the way, just across from the Rockefeller Centre, on the right side of Fifth Avenue is the St Patrick’s Cathedral, its Neo-Gothic style contrasting with and dwarfed by the modern skyscrapers around it.

The shops become more high-end as you get closer to the park, including Saks Fifth Avenue opposite the Rockefeller Centre and the Tiffany & Co flagship store (made famous by the movie Breakfast at Tiffany’s, of course) at 57th Street. And, just so you know, the Museum of Modern Art is on 53rd Street (but I don’t think you’ll have time to go in on this walking tour of New York).

The Plaza Hotel is another New York icon.  If I can continue my movie location theme, it features in The Way We Were and Home Alone 2: Lost in New York. I can highly recommend the cocktails in the Champagne Bar, which is pretty good for people-spotting: I saw Princess Diana’s brother, the Earl of Spencer, there a few years ago.

The Pulitzer Fountain is opposite the hotel and was bequeathed by Joseph Pulitzer, who also established the journalism prizes. There are benches and trees around it, making it a nice spot to sit and rest, if you need it.

Central Park
Once you get to Central Park, I say just have a good wander around – that’s what it is there for! Exploring Central Park is one of the best free things to do in New York. If you want to go as far as the Boathouse, the distance is approximately a mile, so will take 18-20 minutes if you walk non-stop – but of course, there is plenty to distract you in the park!

For example, you might come across talented skaters practising on the paths, or a free performance in the Naumberg Bandshell.  The Bethesda Fountain is a lovely spot, and it is lovely to watch the rowboats on the lake.

Loeb Boathouse has wonderful views over the lake. After completing this self-guided walking tour of Manhattan, you will have earned a rest and some refreshments!

If you still have some energy left, you could continue north and visit the Metropolitan Museum of Art and the Guggenheim Museum, which is on the edge of Central Park. However, I should warn you that my friend and I did this and our feet felt pretty sore by that point!

When you’re ready for dinner, consider one of the unique restaurants in New York City. Or you could end the day with a cocktail in my favourite cocktail bar of all time, the Bemelmans Bar, in the Carlyle Hotel.

Tips For This Walking Tour Of Manhattan
This walking tour of Manhattan is fairly long, so it will pay to be prepared:

Wear comfortable shoes!
Bring water and something waterproof in case the weather changes (& check the forecast in advance)
Respect traffic & use crossings
Be aware the streets of New York can be dirty, cracked and uneven (it’s a gritty city, but I still love it!)
And if this is your first time in New York City, check out these NYC tips for first-time visitors.

When To Go To New York City?
I think the best time to do this walking tour of Manhattan is in the shoulder seasons: March to May and September to October.  I originally did this on the last day of April, and it was glorious!

Don’t be put off visiting New York when it is colder, though: there are lots of things to do in New York in winter and so many things to do in NYC at Christmas.

Where To Stay In New York City
Here are a few places I have stayed in New York recently (in 2024):

The best hotel I’ve stayed in in NYC is, perhaps unsurprisingly, a luxury 5-star hotel: The Wall Street Hotel. It was spacious, plush, very comfortable, and came with all the amenities you could hope for. It was a wonderful oasis of calm to retreat to at the end of a day spent exploring the city. Service was great and the food and cocktails from the lobby bar were also top-tier.

A good 4-star option is the Nolitan, which has a contemporary urban style blending steel, concrete and velvet textures. I had fairly big room, with a balcony and a nice bathroom. They have a restaurant on site, but I opted to eat out in the neighbourhood – there are several good choices nearby on the north end of Mulberry Street. 

For a budget hotel, I was really pleased with my stay at the 3-star Pod 39, in Midtown. It’s hard to find good low-priced accommodation with private bathrooms in Manhattan, but I was impressed with the package here. My single room was clean and space-efficient (small, but not too small). There was free coffee in the bar downstairs in the mornings, and they gave me a discount card for the rooftop bar.

The Last Word
I hope you like this self-guided walking tour of Manhattan! If you follow this route, do let me know how you get on. If you need more inspiration for NYC, check out this list of movies set in New York City, romantic things to do in NYC and unique things to do in NYC.
"""

# Code and Libraries

This is a large block of various functions and other configuration code. You can use it for reference as we walk through the workshop sections, but for now just run the cell.

In [7]:
import boto3
import json
from botocore.exceptions import ClientError
import logging
from enum import Enum
from json import JSONEncoder
import uuid
import csv
import pandas as pd
from pprint import pprint
import time
import io
from difflib import SequenceMatcher

NEWLINE = "\n"
AWS_REGION = json.loads(config)["aws_region"]
endpoint_url = "https://" + json.loads(config)['host'] + ":" + str(json.loads(config)['port'])
neptune_client = boto3.client('neptunedata', endpoint_url=endpoint_url)
bedrock_client = boto3.client("bedrock-runtime", region_name=AWS_REGION)

logger = logging.getLogger(__name__)

def _default(self, obj):
    return getattr(obj.__class__, "__json__", _default.default)(obj)

_default.default = JSONEncoder().default
JSONEncoder.default = _default

context = dict(
    completion_delimiter="||COMPLETE||",
    tuple_delimiter="|~|",
    record_delimiter="##",
    llm_temperature= None,
    llm_top_p = None,
    llm_top_k = None,
    max_tokens = None
)

class NovaMetadata:
    def __init__(self):
        self.input_token_key = "inputTokens"
        self.output_token_key = "outputTokens"
        self.usage_key = "usage"
        self.model_key = "NOVA"

    def get_output_from_response(self, bedrock_response):
        return bedrock_response["output"]["message"]["content"][0]["text"]

class ClaudeMetadata:
    def __init__(self):
        self.input_token_key = "input_tokens"
        self.output_token_key = "output_tokens"
        self.usage_key = "usage"
        self.model_key = "CLAUDE"
        self.anthropic_version = "bedrock-2023-05-31"
        self.max_tokens = 8192
        
    def get_output_from_response(self, bedrock_response):
        return bedrock_response["content"][0]["text"]

def get_model_metadata(model_id):
    match model_id:
        case model_id if model_id.startswith("amazon.nova"):
            return NovaMetadata()
        case model_id if model_id.startswith("us.anthropic.claude") or model_id.startswith("anthropic.claude"):
            return ClaudeMetadata()
        case _:
            return None
            
class CostTracking:
    def __init__(self, model_id):
        self.model_id = model_id
        self.input_tokens = 0
        self.output_tokens = 0
        self.cache_read_input_tokens = 0
        self.cache_write_input_tokens = 0
        self.cost_model = {
            "us.anthropic.claude-3-7-sonnet-20250219-v1:0":
            {
                "inputToken":0.003/1000.0,
                "outputToken":0.015/1000.0,
                "cacheWriteInputToken":0.00375/1000.0,
                "cacheReadInputToken":0.0003/1000.0
            },
            "us.anthropic.claude-3-5-sonnet-20241022-v2:0":
            {
                "inputToken":0.003/1000.0,
                "outputToken":0.015/1000.0,
                "cacheWriteInputToken":0.00375/1000.0,
                "cacheReadInputToken":0.0003/1000.0
            },
            "amazon.nova-pro-v1:0":
            {
                "inputToken":0.0008/1000.0,
                "outputToken":0.00014/1000.0,
                "cacheWriteInputToken":0.000/1000.0, # not listed
                "cacheReadInputToken":0.0002/1000.0
            }
        }
            
        
    def addInputTokenCount(self, token_count):
        self.input_tokens = self.input_tokens + token_count

    def addOutputTokenCount(self, token_count):
        self.output_tokens = self.output_tokens + token_count

    def addCacheReadInputTokenCount(self, token_count):
        self.cache_read_input_tokens = self.input_tokens + token_count

    def addCacheWriteInputTokenCount(self, token_count):
        self.cache_write_input_tokens = self.output_tokens + token_count
                
    def getTokenCount(self):
        return {
            "inputTokens": self.input_tokens,
            "outputTokens": self.output_tokens,
            "cacheReadInputTokens": self.cache_read_input_tokens,
            "cacheWriteInputTokens": self.cache_write_input_tokens
        }
        
    def getInputTokenCost(self):
        return self.input_tokens * self.cost_model[self.model_id]["inputToken"]

    def getOutputTokenCost(self):
        return self.output_tokens * self.cost_model[self.model_id]["outputToken"]
        
    def printVerbose(self):
        inputTokenCost = self.input_tokens * self.cost_model[self.model_id]["inputToken"]
        outputTokenCost = self.output_tokens * self.cost_model[self.model_id]["outputToken"]
        cacheReadTokenCost = self.cache_read_input_tokens * self.cost_model[self.model_id]["cacheReadInputToken"]
        cacheWriteTokenCost = self.cache_write_input_tokens * self.cost_model[self.model_id]["cacheReadInputToken"]
        return f"""
            Input token costs ${inputTokenCost} ({self.input_tokens} tokens)
            Output token costs ${outputTokenCost} ({self.output_tokens} tokens)
            Cache read input token costs ${cacheReadTokenCost} ({self.cache_read_input_tokens} tokens)
            Cache write input token costs ${cacheWriteTokenCost} ({self.cache_write_input_tokens} tokens)
            Total cost of ${inputTokenCost + outputTokenCost + cacheReadTokenCost + cacheWriteTokenCost}
        """


##################################################################################################
#
#  Parts of this code and techniques for use of the LLM were influenced by LightRAG: Simple and Fast Retrieval-Augmented Generation
#  Authors: Zirui Guo and Lianghao Xia and Yanhua Yu and Tu Ao and Chao Huang
#  URL: https://github.com/HKUDS/LightRAG/
#
#
#
##################################################################################################

def run_llm(text, context, bedrock_client, model_id, model_metadata, prompt, do_formatting = True, cost_tracking = None, print_prompts = False):
    
    request_text = None
    
    match model_metadata.model_key: 
        case "NOVA":
            system_list = [
                {
                    "text": prompt["SYSTEM_PROMPT"].format(**context, input_text=text) if do_formatting else prompt["SYSTEM_PROMPT"]
                }
            ]
            message_list = [{"role": "user", "content": [{"text": prompt["USER_PROMPT"].format(**context, input_text=text) if do_formatting else prompt["USER_PROMPT"]}]}]
            inf_params = {
                "max_new_tokens": context["max_tokens"] if context["max_tokens"] is not None else 1000,
                "top_p":context["llm_top_p"] if context["llm_top_p"] is not None else 0.999,
                "top_k":context["llm_top_k"] if context["llm_top_k"] is not None else 20,
                "temperature":context["llm_temperature"] if context["llm_temperature"] is not None else 1
            }
            request_text = {
                "schemaVersion": "messages-v1",
                "messages": message_list,
                "system": system_list,
                "inferenceConfig": inf_params,
            }
        case "CLAUDE":
            message_list = [{"role": "user", "content": prompt["USER_PROMPT"].format(**context, input_text=text) if do_formatting else prompt["USER_PROMPT"]}]
            request_text = {
                "anthropic_version": "bedrock-2023-05-31",
                "max_tokens" : context["max_tokens"] if context["max_tokens"] is not None else 8192,
                "temperature": context["llm_temperature"] if context["llm_temperature"] is not None else 1,
                "top_k": context["llm_top_k"] if context["llm_top_k"] is not None else 250,
                "top_p": context["llm_top_p"] if context["llm_top_p"] is not None else 0.999,
                "system" : prompt["SYSTEM_PROMPT"].format(**context, input_text=text) if do_formatting else prompt["SYSTEM_PROMPT"],
                "messages": message_list
            }
        case _:
            print("Unknown Model Key")
            return None

    # Convert the native request to JSON.
    request = json.dumps(request_text)
    
    response = None
    tries = 0
    
    if print_prompts:
        print(f"System Prompt: {prompt['SYSTEM_PROMPT'].format(**context, input_text=text)}")
        print(f"User Prompt: {prompt['USER_PROMPT'].format(**context, input_text=text)}")

    while response is None:
        try:
            # Invoke the model with the request.
            response = bedrock_client.invoke_model(modelId=model_id, body=request)
            
        except (ClientError, Exception) as exception_obj:
            if exception_obj.response["Error"]["Code"] == "ThrottlingException" and tries <= 5:
                print("I'm being throttled.  Sleeping for 1 second.")
                time.sleep(1)
                tries = tries + 1
            else:
                print(exception_obj.response)
                print(f"ERROR: Can't invoke '{model_id}'. Reason: {exception_obj}")
                return None

    # Decode the response body.
    response_body = json.loads(response["body"].read())
    if not (cost_tracking is None):
        usage_tracking = response_body[model_metadata.usage_key] if not (model_metadata.usage_key is None) else response_body
        cost_tracking.addInputTokenCount(usage_tracking[model_metadata.input_token_key])
        cost_tracking.addOutputTokenCount(usage_tracking[model_metadata.output_token_key])

    return model_metadata.get_output_from_response(response_body)
    
def llm_conversation(message_list, context, bedrock_client, model_id, model_metadata, prompt, do_formatting = True, cost_tracking = None, cache_system_prompt = False, tools=None, print_responses=False):
    inference_config = {}
    additional_model_fields = {}
    
    match model_metadata.model_key: 
        case "NOVA":
            inference_config["temperature"] = context["llm_temperature"] if context["llm_temperature"] is not None else 1
            inference_config["topP"] = context["llm_top_p"] if context["llm_top_p"] is not None else 0.999
            inference_config["maxTokens"] = 1000
        case "CLAUDE":
            inference_config["temperature"] = context["llm_temperature"] if context["llm_temperature"] is not None else 1
            inference_config["topP"] = context["llm_top_p"] if context["llm_top_p"] is not None else 0.999
            inference_config["maxTokens"] = context["max_tokens"] if context["max_tokens"] is not None else 8192
            additional_model_fields["top_k"] = context["llm_top_k"] if context["llm_top_k"] is not None else 250
            
    response = None
    tries = 0
    tool_config = None
    
    system_message = [
        {
            "text": prompt["SYSTEM_PROMPT"].format(**context) if do_formatting else prompt["SYSTEM_PROMPT"]
        }
    ]
    
    if cache_system_prompt:
        system_message.append(
            {"cachePoint": {"type": "default"}}
        )
        
    tool_block = {"tools": tools}

    while response is None and tries <= 5:
        try:
            response = bedrock_client.converse(
                modelId=model_id,
                messages=message_list,
                system=system_message,
                inferenceConfig=inference_config,
                additionalModelRequestFields=additional_model_fields,
                toolConfig=tool_block
            ) if tools is not None else bedrock_client.converse(
                modelId=model_id,
                messages=message_list,
                system=system_message,
                inferenceConfig=inference_config,
                additionalModelRequestFields=additional_model_fields
            )
            if print_responses:
                print(response)
            tries = 0
        except (ClientError, Exception) as exception_obj:
            if tries <= 5:
                print(f"Exception Caught: {exception_obj}")
                tries = tries + 1
#            An error occurred (ThrottlingException) when calling the Converse operation (reached max retries: 4): Too many tokens, please wait before trying again.
#            if exception_obj.response["Error"]["Code"] == "ThrottlingException" and tries <= 5:
#                print("I'm being throttled.  Sleeping for 1 second.")
#                time.sleep(1)
#                tries = tries + 1
#            else:
#                print(exception_obj.response)
#                print(f"ERROR: Can't invoke '{model_id}'. Reason: {exception_obj}")
#                return None
    
    if not (cost_tracking is None) and not (response is None):
        if "usage" in response:
            cost_tracking.addInputTokenCount(response["usage"]["inputTokens"])
            cost_tracking.addOutputTokenCount(response["usage"]["outputTokens"])
            if "cacheReadInputTokens" in response["usage"]:
                cost_tracking.addCacheReadInputTokenCount(response["usage"]["cacheReadInputTokens"])
            if "cacheWriteInputTokens" in response["usage"]:
                cost_tracking.addCacheWriteInputTokenCount(response["usage"]["cacheWriteInputTokens"])
        
    return response
            

key_func = lambda x, y : x.split(delimiter)[y].strip() # our key is the first field
group_by_type_func = lambda x : key_func(x,0)  
group_by_neighborhood_func = lambda x : key_func(x,2)
is_entity = lambda x : is_entity(x,3)
is_entity = lambda x, y : "|" in x and len(x.split("|")) == y

def group_by_key(input_lines, delimiter, key_func):
    grouped_keys = {}
    for line in input_lines:
        if delimiter in line:
            group_name = key_func(line)
            if group_name in grouped_keys:
                grouped_keys[group_name] = grouped_keys[group_name] + 1
            else:
                grouped_keys[group_name] = 1
    return grouped_keys
    
def list_to_dataframe(data, delimiter='|',column_names=None):
    if isinstance(data, str):
        data = io.StringIO(data)
    else:
        data = io.StringIO(NEWLINE.join(data))
    return pd.read_csv(data, sep=delimiter,header=None,names=column_names)
    
def string_compare(df, col1, col2):
    return SequenceMatcher(None, df[col1],df[col2]).ratio()

def generate_diff_df(entity_df, compare_df):
    cross_df = entity_df.merge(compare_df, how='cross')
    cross_df["name_similarity_ratio"] = cross_df.apply(string_compare, args=('name_x','name_y'), axis=1)
    grouped_df = cross_df.groupby(["type_x",'name_x','neighborhood_x'])
    max_index = grouped_df["name_similarity_ratio"].idxmax()
    return cross_df[cross_df.index.isin(max_index) & ((cross_df["name_similarity_ratio"] < 1.0) | (cross_df["type_x"] != cross_df["type_y"]) | (cross_df["neighborhood_x"] != cross_df["neighborhood_y"]))]
    

class SupportedModels(Enum):
    ANTHROPIC_CLAUDE_3_7_SONNET = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
    ANTHROPIC_CLAUDE_3_5_SONNET_v2 = "us.anthropic.claude-3-5-sonnet-20241022-v2:0"
    AMAZON_NOVA_PRO_1_0 = "amazon.nova-pro-v1:0"


# The Basics of Entity Extraction

Notes for this section:  Remember that LLMs are non-deterministic, so the instructions will assume the experience for most people, but it is possible your experience may vary.  If you don't see the expected result, except where noted, rerunning the step until you do get the expected result should work.

## Let's start with a very basic prompt to extract the list of entities mentioned in this walking tour guide

In [8]:
# We will utilize the Anthropic Claude 3.5 Sonnet V2 Foundational Model in this part of the workshop
model = SupportedModels.ANTHROPIC_CLAUDE_3_5_SONNET_v2
model_id = model.value
model_metadata = get_model_metadata(model_id)

reference_text = WALKING_TOUR_TEXT

# This object will be used to track our costs as calculated from Bedrock's output.
costs = CostTracking(model_id)

# Each LLM call contains two prompts.
PROMPT = {}
# First, the System instructions describing the LLMs role and behavior.
PROMPT["SYSTEM_PROMPT"] = """
-Goal-
Given the input text, extract a list of entities and group them by the type of entity they are.  
List these entities for me.  
Output one entity per line.
"""
# Second, the user's message to the model.
PROMPT["USER_PROMPT"] = """
Text: {input_text}
"""

response_text = run_llm(reference_text,context,bedrock_client,model_id,model_metadata,PROMPT,cost_tracking=costs)

print(costs.printVerbose())
print(response_text)



            Input token costs $0.012909 (4303 tokens)
            Output token costs $0.005189999999999999 (346 tokens)
            Cache read input token costs $0.0 (0 tokens)
            Cache write input token costs $0.0 (0 tokens)
            Total cost of $0.018099
        
Here are the entities grouped by type:

NEIGHBORHOODS/AREAS:
Downtown Manhattan
Midtown
Upper East Side
Greenwich Village
West Village
East Village
Little Germany

PARKS/GARDENS:
Central Park
Washington Square Park
Madison Square Park
Bryant Park
St Luke in the Fields Garden
Tomkins Square Park
Union Square Park

BUILDINGS/LANDMARKS:
Flatiron Building
Empire State Building
New York Public Library
Rockefeller Centre
Plaza Hotel
Chrysler Building
St Patrick's Cathedral
Metropolitan Museum of Art
Guggenheim Museum
Morgan Library & Museum
Jefferson Market Library
Radio City Hall
Grand Central Station
The Carlyle Hotel
Grace Church
Metropolitan Life Insurance Company Building
New York Life Building
Stonewall Inn

S

<div class="alert alert-block alert-warning">If you see an error stating 
<pre>Can't invoke 'us.anthropic.claude-3-5-sonnet-20241022-v2:0'. Reason: An error occurred (AccessDeniedException) when calling the InvokeModel operation: You don't have access to the model with the specified model ID.</pre>
please go to <a href="https://us-west-2.console.aws.amazon.com/bedrock/home?region=us-west-2#/modelaccess">Bedrock Model Access</a>, choose "Modify Model Access", and selected Claude 3.5 Sonnet v2 and Claude 3.7 Sonnet.</div>

<br>This is a good start, but it will be very difficult for a machine to read this. Let's ask the LLM to provide the results in a **more machine-readable format.**

In [9]:
costs = CostTracking(model_id)
PROMPT = {}
PROMPT["SYSTEM_PROMPT"] = """
-Goal-
Given the input text, extract a list of entities and determine the type of entity they are.  
List these entities for me, one per line, in the format of:
type | entity
"""

PROMPT["USER_PROMPT"] = """
Text: {input_text}
"""
    
response_text = run_llm(reference_text,context,bedrock_client,model_id,model_metadata,PROMPT,cost_tracking=costs)

print(costs.printVerbose())
print(response_text)


            Input token costs $0.012918 (4306 tokens)
            Output token costs $0.00663 (442 tokens)
            Cache read input token costs $0.0 (0 tokens)
            Cache write input token costs $0.0 (0 tokens)
            Total cost of $0.019548
        
Here are the entities from the text and their types:

location | Greenwich Village
location | Manhattan
location | Central Park
location | Upper East Side
location | Washington Square Park
building | Flatiron Building
location | Madison Square Park
building | Empire State Building
building | New York Public Library
location | Fifth Avenue
building | Rockefeller Centre
building | Plaza Hotel
landmark | Pulitzer Fountain
building | St Luke in the Fields Garden
location | Hudson Street
location | Barrow Street
location | West Village
location | Grove Street
landmark | Friends building
location | Christopher Park
business | Stonewall Inn
business | Joe's Pizza
building | Jefferson Market Library
location | Perry Street
busines

The list above probably has some locations like neighborhoods and streets, which we don't want to consider as entities in our graph.  
Let's add some instructions to guide it away from that, such as telling it we only want businesses, museums, and landmarks to be considered landmarks.
We are also going to ask it to add an additional field that is going to be challenging for it...the NYC neighborhood each entity is located in.

In [10]:
costs = CostTracking(model_id)
PROMPT = {}
PROMPT["SYSTEM_PROMPT"] = """
-Goal-
Given the input text, extract a list of entities and determine the type of entity they are.
Entities are businesses, museums, or landmarks, but not neighborhoods or streets.
If you know what New York City neighborhood an entity is located in, then list it in the neighborhood field.
List these entities for me, one per line, in the format of:
type | entity | neighborhood

"""

PROMPT["USER_PROMPT"] = """
Text: {input_text}
"""
    
response_text = run_llm(reference_text,context,bedrock_client,model_id,model_metadata,PROMPT,cost_tracking=costs)

print(costs.printVerbose())
print(response_text)


            Input token costs $0.013041 (4347 tokens)
            Output token costs $0.004515 (301 tokens)
            Cache read input token costs $0.0 (0 tokens)
            Cache write input token costs $0.0 (0 tokens)
            Total cost of $0.017556000000000002
        
Here are the entities from the text:

landmark | Washington Square Arch | Greenwich Village
hotel | Plaza Hotel | Midtown
museum | Metropolitan Museum of Art | Upper East Side
museum | Guggenheim Museum | Upper East Side
landmark | Empire State Building | Midtown
landmark | Flatiron Building | Flatiron District
landmark | New York Public Library | Midtown
landmark | Rockefeller Centre | Midtown
museum | Museum of Modern Art | Midtown
landmark | St Patrick's Cathedral | Midtown
business | Tiffany & Co | Midtown
business | Saks Fifth Avenue | Midtown
business | Radio City Hall | Midtown
business | Gotham Bar & Grill | Greenwich Village
business | Levain Bakery | Chelsea
business | Eataly | Flatiron District
busi

In most cases you'll find that the LLM took our list of entities as a literal list and your entities are probably all listed as "landmark","business",or "museum".  
Let's change those names to be all upper case, so we can easily distinguish what will be our node labels.

In [11]:
costs = CostTracking(model_id)
PROMPT = {}
PROMPT["SYSTEM_PROMPT"] = """
-Goal-
Given the input text, extract a list of entities and determine the type of entity they are.
Entities are "BUSINESS", "MUSEUM", or "LANDMARK", but not neighborhoods or streets.
If you know what New York City neighborhood an entity is located in, then list it in the neighborhood field.
List these entities for me, one per line, in the format of:
type | entity | neighborhood

"""

PROMPT["USER_PROMPT"] = """
Text: {input_text}
"""
    
response_text = run_llm(reference_text,context,bedrock_client,model_id,model_metadata,PROMPT,cost_tracking=costs)

print(costs.printVerbose())
print(response_text)



            Input token costs $0.013059000000000001 (4353 tokens)
            Output token costs $0.0032999999999999995 (220 tokens)
            Cache read input token costs $0.0 (0 tokens)
            Cache write input token costs $0.0 (0 tokens)
            Total cost of $0.016359000000000002
        
Here are the relevant entities from the text:

LANDMARK | Washington Square Arch | Greenwich Village
LANDMARK | Flatiron Building | Flatiron District
LANDMARK | Empire State Building | Midtown
LANDMARK | Chrysler Building | Midtown
MUSEUM | Morgan Library & Museum | Midtown
LANDMARK | New York Public Library | Midtown
BUSINESS | Radio City Hall | Midtown
LANDMARK | St Patrick's Cathedral | Midtown
BUSINESS | Plaza Hotel | Midtown
MUSEUM | Museum of Modern Art | Midtown
BUSINESS | Loeb Boathouse | Upper East Side
MUSEUM | Metropolitan Museum of Art | Upper East Side
MUSEUM | Guggenheim Museum | Upper East Side
BUSINESS | Bemelmans Bar | Upper East Side
BUSINESS | The Wall Street Hotel |

Let's gather some summary statistics by entity type and neighborhood so we can compare later and see the effects various techniques have.  
You may be noticing already that some of these neighborhoods aren't really considered NYC neighborhoods.

In [12]:
lines = response_text.splitlines(keepends=True)
delimiter = "|" 

print(group_by_key(lines, delimiter, group_by_type_func))
print(group_by_key(lines, delimiter, group_by_neighborhood_func))

# save these results for later
single_pass_entities = set([line for line in response_text.splitlines() if is_entity(line,3)])

{'LANDMARK': 6, 'MUSEUM': 4, 'BUSINESS': 7}
{'Greenwich Village': 1, 'Flatiron District': 1, 'Midtown': 9, 'Upper East Side': 4, 'Financial District': 1, 'Nolita': 1}


# Strategy: Prompt Looping

You might suspect there are more than the 20 or so entities we extracted before in the walking tour and I suspect you will be correct.
Here we are taking a new approach where we will use the same prompt, but ask the LLM up to 10 times to "take another look" and see if you missed anything so far.
Each time we will give it the list of entities it had found up to that point in previous iterations.  Let's see how this goes...

In [13]:
costs = CostTracking(model_id)
PROMPT = {}

entities = []
more_entities = True
max_iterations = 10
iterations = 0

while more_entities and iterations < max_iterations: 
    PROMPT["SYSTEM_PROMPT"] = """
    -Goal-
    Given the input text, extract a list of entities and determine the type of entity they are.
    Entities are "BUSINESS", "MUSEUM", or "LANDMARK", but not neighborhoods or streets.
    If you know what New York City neighborhood an entity is located in, then list it in the neighborhood field.
    List these entities for me, one per line, in the format of:
    type | entity | neighborhood

    There may be entities you already identified in the list below:
    {entities}
    Please check this text for entities that were not already identified, and please list them if there are.
    """

    PROMPT["USER_PROMPT"] = """
    Text: {input_text}
    """

    # The context object stores parameters and strings that will be passed into the LLM call. 
    # In this case, the text here will be substituted into the SYSTEM prompt where the placeholder {entities} exists.
    context["entities"] = NEWLINE.join(entities)

    response_text = run_llm(reference_text,context,bedrock_client,model_id,model_metadata,PROMPT,cost_tracking=costs,print_prompts=False)

    new_entities = [line for line in response_text.splitlines() if is_entity(line,3)]

    entities = entities + new_entities
    new_entity_count = len(new_entities)
    more_entities = new_entity_count > 0
    iterations = iterations + 1
    print(f"Iteration {iterations}. Found {new_entity_count} new entities. {len(entities)} entities total now")
    
print(costs.printVerbose())
print(NEWLINE.join(entities))

Iteration 1. Found 34 new entities. 34 entities total now
Iteration 2. Found 12 new entities. 46 entities total now
Iteration 3. Found 7 new entities. 53 entities total now
Iteration 4. Found 3 new entities. 56 entities total now
Iteration 5. Found 1 new entities. 57 entities total now
Iteration 6. Found 3 new entities. 60 entities total now
Iteration 7. Found 4 new entities. 64 entities total now
Iteration 8. Found 0 new entities. 64 entities total now

            Input token costs $0.11934 (39780 tokens)
            Output token costs $0.02463 (1642 tokens)
            Cache read input token costs $0.0 (0 tokens)
            Cache write input token costs $0.0 (0 tokens)
            Total cost of $0.14397
        
LANDMARK | Washington Square Arch | Greenwich Village
LANDMARK | Washington Square Fountain | Greenwich Village
BUSINESS | Stonewall Inn | Greenwich Village
BUSINESS | Joe's Pizza | Greenwich Village
LANDMARK | Jefferson Market Library | Greenwich Village
BUSINESS | Angelik

And again we can look at some statistics about how these entities by type and neighborhood.  

In [14]:
delimiter = "|"

print(group_by_key(entities, delimiter, group_by_type_func))
print(group_by_key(entities, delimiter, group_by_neighborhood_func))

{'LANDMARK': 30, 'BUSINESS': 28, 'MUSEUM': 6}
{'Greenwich Village': 14, 'East Village': 7, 'Flatiron District': 6, 'Midtown': 23, 'Central Park': 4, 'Upper East Side': 4, 'Financial District': 1, 'Nolita': 1, 'Union Square': 4}


Sometimes the LLM might duplicate entries across iterations, so let's convert into a set to remove duplicates

In [15]:
unique_entities = set(entities)
print(group_by_key(unique_entities, delimiter, group_by_type_func))
print(group_by_key(unique_entities, delimiter, group_by_neighborhood_func))

{'BUSINESS': 28, 'LANDMARK': 30, 'MUSEUM': 4}
{'Nolita': 1, 'Midtown': 21, 'Union Square': 4, 'Upper East Side': 4, 'East Village': 7, 'Flatiron District': 6, 'Greenwich Village': 14, 'Central Park': 4, 'Financial District': 1}


You can see we extracted many more entities using this looping strategy rather than the single pass.<br>
**However it is more likely with this looping strategy that the LLM may have hallucinated or taken some liberties in trying to please us with answers.**<br>
So let's have the LLM fact check itself.  We will print a list of what entities it confirms and which it believes are duplicate.

In [16]:
costs = CostTracking(model_id)
PROMPT = {}

PROMPT["SYSTEM_PROMPT"] = """
-Goal-
You are a quality assurance reviewer who accepts two inputs: a list of entities, and text which those entities 
were extracted from.  The entities come in the form of a delimited list where each line is a single entity 
and it has three categories of information: type of entity, entity name, and NYC neighborhood where it resides.
The entities are presented in the following format:
type | entity | neighborhood
Your job is to ensure each entity is mentioned in the text and not a hallucination from the original extraction.
List the entities that actually exist, one per line, in the same format:
type | entity | neighborhood

"""

PROMPT["USER_PROMPT"] = """
Entities:
{entity_list}
Text: 
{input_text}
"""

context["entity_list"] = NEWLINE.join(unique_entities)

response_text = run_llm(reference_text,context,bedrock_client,model_id,model_metadata,PROMPT,cost_tracking=costs)

raw_confirmed_entities = [line for line in response_text.splitlines() if is_entity(line,3)]
confirmed_entities = set(raw_confirmed_entities)

print(costs.printVerbose())
# print(response_text)


removed_entities = unique_entities.difference(set(confirmed_entities))
print(f"Confirmed entities: {NEWLINE} {NEWLINE.join(confirmed_entities)}")
print(f"Removed entities: {NEWLINE} {NEWLINE.join(removed_entities)}")


            Input token costs $0.015528 (5176 tokens)
            Output token costs $0.010289999999999999 (686 tokens)
            Cache read input token costs $0.0 (0 tokens)
            Cache write input token costs $0.0 (0 tokens)
            Total cost of $0.025818
        
Confirmed entities: 
 BUSINESS | The Nolitan | Nolita
LANDMARK | St Patrick's Cathedral | Midtown
LANDMARK | George Washington Statue | Union Square
LANDMARK | Carrie Bradshaw's Apartment | Greenwich Village
BUSINESS | Champagne Bar (Plaza Hotel) | Midtown
BUSINESS | Rose Reading Room | Midtown
BUSINESS | Bemelmans Bar | Upper East Side
LANDMARK | Flatiron Building | Flatiron District
LANDMARK | Friends Apartment Building | Greenwich Village
BUSINESS | Loeb Boathouse | Central Park
BUSINESS | Pod 39 | Midtown
BUSINESS | Veselka | East Village
BUSINESS | Saks Fifth Avenue | Midtown
MUSEUM | Guggenheim Museum | Upper East Side
LANDMARK | Empire State Building | Midtown
LANDMARK | Grace Church | East Village
LAND

And let's see some summary statistics across both those groups. You should see many more results than the first pass.

In [17]:
delimiter = "|"

print("Confirmed:")
print(group_by_key(confirmed_entities, delimiter, group_by_type_func))
print(group_by_key(confirmed_entities, delimiter, group_by_neighborhood_func))
print("Removed:")
print(group_by_key(removed_entities, delimiter, group_by_type_func))
print(group_by_key(removed_entities, delimiter, group_by_neighborhood_func))


Confirmed:
{'BUSINESS': 22, 'LANDMARK': 28, 'MUSEUM': 4}
{'Nolita': 1, 'Midtown': 19, 'Union Square': 3, 'Greenwich Village': 12, 'Upper East Side': 3, 'Flatiron District': 6, 'Central Park': 3, 'East Village': 6, 'Financial District': 1}
Removed:
{'BUSINESS': 6, 'LANDMARK': 2}
{'East Village': 1, 'Greenwich Village': 2, 'Union Square': 1, 'Upper East Side': 1, 'Midtown': 2, 'Central Park': 1}


There are likely some subtle differences between the records in the two.  Let's compare the results between our single pass extraction and the looped extraction where they are almost the same but different.<br>
NOTE:  There may be some cases where an entire entity was missed. In those cases, it is showing the name that is most similar, but the similarity score will be pretty low.

In [18]:
single_pass_entities_df = list_to_dataframe(single_pass_entities,delimiter='|',column_names=["type","name","neighborhood"]).sort_values(by="name")
confirmed_entities_df = list_to_dataframe(confirmed_entities,delimiter='|',column_names=["type","name","neighborhood"]).sort_values(by="name")

single_pass_diff_df = generate_diff_df(single_pass_entities_df, confirmed_entities_df)

print(single_pass_diff_df)


        type_x                     name_x    neighborhood_x     type_y  \
294  BUSINESS             Loeb Boathouse    Upper East Side  BUSINESS    
421    MUSEUM    Morgan Library & Museum            Midtown    MUSEUM    
547  BUSINESS                Plaza Hotel            Midtown  BUSINESS    
682  BUSINESS            Radio City Hall            Midtown  LANDMARK    

                            name_y neighborhood_y  name_similarity_ratio  
294                Loeb Boathouse    Central Park               1.000000  
421   The Morgan Library & Museum         Midtown               0.925926  
547   Champagne Bar (Plaza Hotel)         Midtown               0.619048  
682               Radio City Hall         Midtown               1.000000  


You probably see some subtle differences in name, how things are labeled business or landmark, or what neighborhood they appear in.  Let's give more specific entity definitions into the first prompt to improve this outcome (e.g., let's start to define an ontology for the LLM to follow).

In [19]:
costs = CostTracking(model_id)
PROMPT = {}
PROMPT["SYSTEM_PROMPT"] = """
-Goal-
Given the input text, extract a list of entities and determine the type of entity they are.
The valid list of entities are: 
    "BUSINESS": a location where you can purchase items or services, such as a restaurant or store.
    "MUSEUM": a location where you view historical, scientific, artistic, or cultural objects, but they are not for sale
    "LANDMARK": a physical structure or location that is well recognized, may have historical or cultural significance, and may contain businesses, but is not a business itself. 

Classify libraries as LANDMARK.
Neighborhoods and streets as not entities.
If you know what New York City neighborhood an entity is located in, then list it in the neighborhood field.
List these entities for me, one per line, in the format of:
type | entity | neighborhood

"""

PROMPT["USER_PROMPT"] = """
Text: {input_text}
"""
    
response_text = run_llm(reference_text,context,bedrock_client,model_id,model_metadata,PROMPT,cost_tracking=costs)

print(costs.printVerbose())
print(response_text)

lines = response_text.splitlines(keepends=True)
delimiter = "|" 

print(group_by_key(lines, delimiter, group_by_type_func))
print(group_by_key(lines, delimiter, group_by_neighborhood_func))

# save these results for later
single_pass_entities_improved_prompt = set([line for line in response_text.splitlines() if is_entity(line,3)])


            Input token costs $0.013326000000000001 (4442 tokens)
            Output token costs $0.004875 (325 tokens)
            Cache read input token costs $0.0 (0 tokens)
            Cache write input token costs $0.0 (0 tokens)
            Total cost of $0.018201000000000002
        
Let me extract the entities from this text:

LANDMARK | Washington Square Park | Greenwich Village
LANDMARK | Washington Square Arch | Greenwich Village
LANDMARK | One Fifth Avenue | Greenwich Village
LANDMARK | Empire State Building | Midtown
LANDMARK | Flatiron Building | Flatiron District
LANDMARK | New York Public Library | Midtown
LANDMARK | St. Patrick's Cathedral | Midtown
LANDMARK | Plaza Hotel | Upper East Side
LANDMARK | Pulitzer Fountain | Upper East Side
LANDMARK | Rockefeller Centre | Midtown
LANDMARK | Radio City Hall | Midtown
LANDMARK | Chrysler Building | Midtown
LANDMARK | Grand Central Station | Midtown
LANDMARK | Jefferson Market Library | Greenwich Village
BUSINESS | Joe's Pizz

In [20]:
single_pass_entities_improved_prompt_df = list_to_dataframe(single_pass_entities_improved_prompt,delimiter='|',column_names=["type","name","neighborhood"]).sort_values(by="name")

improved_prompt_diff_df = generate_diff_df(single_pass_entities_improved_prompt_df, confirmed_entities_df)

print(improved_prompt_diff_df)

         type_x                     name_x      neighborhood_x     type_y  \
0     BUSINESS      230 Fifth Rooftop Bar    Flatiron District  BUSINESS    
71    BUSINESS                      B Cup         East Village  LANDMARK    
670   BUSINESS              Levain Bakery              Chelsea  BUSINESS    
925   LANDMARK                Plaza Hotel      Upper East Side  BUSINESS    
1017  LANDMARK          Pulitzer Fountain      Upper East Side  BUSINESS    
1174  LANDMARK    St. Patrick's Cathedral              Midtown  LANDMARK    

                             name_y      neighborhood_y  name_similarity_ratio  
0            230 Fifth Rooftop Bar              Midtown               1.000000  
71                    Grace Church         East Village               0.476190  
670                  Levain Bakery    Flatiron District               1.000000  
925    Champagne Bar (Plaza Hotel)              Midtown               0.619048  
1017         The Pulitzer Fountain              Midtown

It is very likely you notice some examples where the name is slightly different between each run. Our next strategy will help resolve this issue, called multi-shot prompting.

## Strategy: Multi-Shot Prompts

We were using zero-shot prompting before. This means we were asking the LLM to perform a (fairly) complex task without examples.
You likely saw an example where one attempt extracted an entity "The Plaza Hotel" or "The Wall Street Hotel" and the other extracted the same as "Plaza Hotel" or "Wall Street Hotel".
Let's see what a multi-shot prompt example looks like to help solve how the word "The" is captured as in "The Plaza Hotel" or "The Wall Street Hotel".

In [21]:
costs = CostTracking(model_id)
PROMPT = {}
PROMPT["SYSTEM_PROMPT"] = """
-Goal-
Given the input text, extract a list of entities and determine the type of entity they are.
The valid list of entities are: 
    "BUSINESS": a location where you can purchase items or services, such as a restaurant or store.
    "MUSEUM": a location where you view historical, scientific, artistic, or cultural objects, but they are not for sale.
    "LANDMARK": a physical structure or location that is well recognized, may have historical or cultural significance, and may contain businesses, but is not a business itself. 

Classify libraries as LANDMARK.
Neighborhoods and streets as not entities.
If you know what New York City neighborhood an entity is located in, then list it in the neighborhood field.
List these entities for me, one per line, in the format of:
type | entity | neighborhood

Below are a few examples of how to extract the entities.
==========
Example 1:
Text: 
Visit The Bank of New York Mellon at 240 Greenwich Street in the Tribeca neighborhood of Manhattan
Record:
BUSINESS | The Bank of New York Mellon | Tribeca
==========
Example 2:
Text: 
The Empire State Building at 20 W 34th St., New York, NY 10001 is a landmark in the Midtown neighborhood of Manhattan. Visit the Empire State Building!
Record:
LANDMARK | Empire State Building | Midtown
==========
Example 3:
Text:
The Waldorf Astoria in Midtown is a lovely hotel. I urge you to stay at The Waldorf Astoria.
Record:
BUSINESS | The Waldorf Astoria | Midtown
"""

PROMPT["USER_PROMPT"] = """
Text: {input_text}
"""
    
response_text = run_llm(reference_text,context,bedrock_client,model_id,model_metadata,PROMPT,cost_tracking=costs)

print(costs.printVerbose())
print(response_text)

lines = response_text.splitlines(keepends=True)
delimiter = "|" 

print(group_by_key(lines, delimiter, group_by_type_func))
print(group_by_key(lines, delimiter, group_by_neighborhood_func))

# save these results for later
single_pass_entities_multi_shot = set([line for line in response_text.splitlines() if is_entity(line,3)])


            Input token costs $0.013917 (4639 tokens)
            Output token costs $0.007424999999999999 (495 tokens)
            Cache read input token costs $0.0 (0 tokens)
            Cache write input token costs $0.0 (0 tokens)
            Total cost of $0.021342
        
Here are the entities from the text:

LANDMARK | St Luke in the Fields Garden | West Village
LANDMARK | Washington Square Park | Greenwich Village
LANDMARK | Christopher Park | Greenwich Village
BUSINESS | Stonewall Inn | Greenwich Village
BUSINESS | Joe's Pizza | Greenwich Village
LANDMARK | Jefferson Market Library | Greenwich Village
LANDMARK | Tompkins Square Park | East Village
BUSINESS | The Maiden Lane | East Village
BUSINESS | B Cup | East Village
BUSINESS | Veselka | East Village
LANDMARK | Grace Church | East Village
LANDMARK | Washington Square Arch | Greenwich Village
BUSINESS | Gotham Bar & Grill | Greenwich Village
LANDMARK | Union Square Park | Union Square
BUSINESS | Levain Bakery | Chelsea
LAN

In [22]:
single_pass_entities_multi_shot_df = list_to_dataframe(single_pass_entities_multi_shot,delimiter='|',column_names=["type","name","neighborhood"]).sort_values(by="name")

multi_shot_diff_df = generate_diff_df(single_pass_entities_multi_shot_df, confirmed_entities_df)

print(multi_shot_diff_df)

         type_x                          name_x  \
71    BUSINESS                           B Cup    
275   LANDMARK                    Central Park    
886   BUSINESS                   Levain Bakery    
1123    MUSEUM         Morgan Library & Museum    
1341  LANDMARK               Pulitzer Fountain    
1497  LANDMARK    St Luke in the Fields Garden    
1735  BUSINESS                 The Plaza Hotel    

                        neighborhood_x     type_y  \
71                        East Village  LANDMARK    
275    Upper East Side/Upper West Side  LANDMARK    
886                            Chelsea  BUSINESS    
1123                           Midtown    MUSEUM    
1341                           Midtown  BUSINESS    
1497                      West Village  LANDMARK    
1735                           Midtown  BUSINESS    

                              name_y      neighborhood_y  \
71                     Grace Church         East Village   
275                     Bryant Park           

Hopefully you don't see any more differences where "The" prefixes names or not.  We still have some inconsistencies between BUSINESS and LANDMARK, maybe we need to further refine that definition. We are still haunted by inconsistent neighborhood listings though.  For example, you may see something like the "Loeb Boathouse" identified as being in the "Central Park" neighborhood, but most sources consider Central Park not to be a neighborhood but rather part of the Upper East Side neighborhood. Let's address that next by passing in a list of neighborhoods for the LLM to choose from.

## Strategy: Domain Constraints

OK, we sort of already used this strategy already when we gave it a list of entities we wanted to choose from, but we'll be explicit here as well with providing a list of neighborhoods to choose from.

In [23]:
neighborhood_list = ["East Village","Financial District","Flatiron District","Greenwich Village","Midtown","Murray Hill",
"Nolita","Union Square","Upper East Side","West Village"]

costs = CostTracking(model_id)
context["neighborhood_list"] = ",".join(neighborhood_list)
PROMPT = {}
PROMPT["SYSTEM_PROMPT"] = """
-Goal-
Given the input text, extract a list of entities and determine the type of entity they are.
The valid list of entities are: 
    "BUSINESS": a location where you can purchase items or services, such as a restaurant or store.
    "MUSEUM": a location where you view historical, scientific, artistic, or cultural objects, but they are not for sale.
    "LANDMARK": a physical structure or location that is well recognized, may have historical or cultural significance, and may contain businesses, but is not a business itself. 

Classify libraries as LANDMARK.
Neighborhoods and streets as not entities.
If you know what New York City neighborhood an entity is located in, then list it in the neighborhood field. 
Use this list of neighborhoods to choose from: {neighborhood_list}
List these entities for me, one per line, in the format of:
type | entity | neighborhood

Below are a few examples of how to extract the entities.
==========
Example 1:
Text: 
Visit The Bank of New York Mellon at 240 Greenwich Street in the Tribeca neighborhood of Manhattan
Record:
BUSINESS | The Bank of New York Mellon | Tribeca
==========
Example 2:
Text: 
The Empire State Building at 20 W 34th St., New York, NY 10001 is a landmark in the Midtown neighborhood of Manhattan. Visit the Empire State Building!
Record:
LANDMARK | Empire State Building | Midtown
==========
Example 3:
Text:
The Waldorf Astoria in Midtown is a lovely hotel. I urge you to stay at The Waldorf Astoria.
Record:
BUSINESS | The Waldorf Astoria | Midtown

"""

PROMPT["USER_PROMPT"] = """
Text: {input_text}
"""
    
response_text = run_llm(reference_text,context,bedrock_client,model_id,model_metadata,PROMPT,cost_tracking=costs)

print(costs.printVerbose())
print(response_text)

lines = response_text.splitlines(keepends=True)
delimiter = "|" 

print(group_by_key(lines, delimiter, group_by_type_func))
print(group_by_key(lines, delimiter, group_by_neighborhood_func))

# save these results for later
single_pass_entities_neighborhoods = set([line for line in response_text.splitlines() if is_entity(line,3)])


            Input token costs $0.014058000000000001 (4686 tokens)
            Output token costs $0.005939999999999999 (396 tokens)
            Cache read input token costs $0.0 (0 tokens)
            Cache write input token costs $0.0 (0 tokens)
            Total cost of $0.019998000000000002
        
Here are the entities from the text, organized by type:

LANDMARK | Washington Square Arch | Greenwich Village
LANDMARK | Empire State Building | Midtown
LANDMARK | New York Public Library | Midtown
LANDMARK | Rockefeller Centre | Midtown
LANDMARK | St Patrick's Cathedral | Midtown
LANDMARK | Jefferson Market Library | Greenwich Village
LANDMARK | Grace Church | East Village
LANDMARK | Chrysler Building | Midtown
LANDMARK | Grand Central Station | Midtown
LANDMARK | Radio City Hall | Midtown
LANDMARK | St Luke in the Fields Garden | West Village
LANDMARK | Metropolitan Museum of Art | Upper East Side
LANDMARK | Guggenheim Museum | Upper East Side

BUSINESS | Joe's Pizza | Greenwich Vill

In [24]:
single_pass_entities_neighborhoods_df = list_to_dataframe(single_pass_entities_neighborhoods,delimiter='|',column_names=["type","name","neighborhood"]).sort_values(by="name")

neighborhoods_diff_df = generate_diff_df(single_pass_entities_neighborhoods_df, confirmed_entities_df)

print(neighborhoods_diff_df)

         type_x                          name_x      neighborhood_x  \
0     BUSINESS           230 Fifth Rooftop Bar    Flatiron District   
71    BUSINESS                           B Cup         East Village   
340   BUSINESS              Gotham Bar & Grill         Union Square   
505   LANDMARK               Guggenheim Museum      Upper East Side   
726   BUSINESS                  Loeb Boathouse      Upper East Side   
783   LANDMARK      Metropolitan Museum of Art      Upper East Side   
853     MUSEUM         Morgan Library & Museum          Murray Hill   
1227  LANDMARK    St Luke in the Fields Garden         West Village   
1411  BUSINESS                 The Plaza Hotel              Midtown   

         type_y                          name_y      neighborhood_y  \
0     BUSINESS           230 Fifth Rooftop Bar              Midtown   
71    LANDMARK                    Grace Church         East Village   
340   BUSINESS              Gotham Bar & Grill    Greenwich Village   
505  

Now let's run our looping prompt again with our improved prompt and compare the results.

In [25]:
costs = CostTracking(model_id)
PROMPT = {}

entities = []
more_entities = True
max_iterations = 10
iterations = 0

while more_entities and iterations < max_iterations: 
    PROMPT["SYSTEM_PROMPT"] = """
    -Goal-
    Given the input text, extract a list of entities and determine the type of entity they are.
    The valid list of entities are: 
        "BUSINESS": a location where you can purchase items or services, such as a restaurant or store.
        "MUSEUM": a location where you view historical, scientific, artistic, or cultural objects, but they are not for sale.
        "LANDMARK": a physical structure or location that is well recognized, may have historical or cultural significance, and may contain businesses, but is not a business itself. 

    Classify libraries as LANDMARK.
    Neighborhoods and streets as not entities.
    If you know what New York City neighborhood an entity is located in, then list it in the neighborhood field. 
    Use this list of neighborhoods to choose from: {neighborhood_list}
    List these entities for me, one per line, in the format of:
    type | entity | neighborhood

    Below are a few examples of how to extract the entities.
    ==========
    Example 1:
    Text: 
    Visit The Bank of New York Mellon at 240 Greenwich Street in the Tribeca neighborhood of Manhattan
    Record:
    BUSINESS | The Bank of New York Mellon | Tribeca
    ==========
    Example 2:
    Text: 
    The Empire State Building at 20 W 34th St., New York, NY 10001 is a landmark in the Midtown neighborhood of Manhattan. Visit the Empire State Building!
    Record:
    LANDMARK | Empire State Building | Midtown
    ==========
    Example 3:
    Text:
    The Waldorf Astoria in Midtown is a lovely hotel. I urge you to stay at The Waldorf Astoria.
    Record:
    BUSINESS | The Waldorf Astoria | Midtown

    There may be entities you already identified in the list below:
    {entities}
    Please check this text for entities that were not already identified, and please list them if there are.
    """

    PROMPT["USER_PROMPT"] = """
    Text: {input_text}
    """

    context["entities"] = NEWLINE.join(entities)

    response_text = run_llm(reference_text,context,bedrock_client,model_id,model_metadata,PROMPT,cost_tracking=costs,print_prompts=False)

    new_entities = [line for line in response_text.splitlines() if is_entity(line,3)]

    entities = entities + new_entities
    new_entity_count = len(new_entities)
    more_entities = new_entity_count > 0
    iterations = iterations + 1
    print(f"Iteration {iterations}. Found {new_entity_count} new entities. {len(entities)} entities total now")
    
print(costs.printVerbose())
print(NEWLINE.join(entities))

Iteration 1. Found 24 new entities. 24 entities total now
Iteration 2. Found 14 new entities. 38 entities total now
Iteration 3. Found 10 new entities. 48 entities total now
Iteration 4. Found 3 new entities. 51 entities total now
Iteration 5. Found 1 new entities. 52 entities total now
Iteration 6. Found 3 new entities. 55 entities total now
Iteration 7. Found 3 new entities. 58 entities total now
Iteration 8. Found 3 new entities. 61 entities total now
Iteration 9. Found 3 new entities. 64 entities total now
Iteration 10. Found 2 new entities. 66 entities total now

            Input token costs $0.159399 (53133 tokens)
            Output token costs $0.031334999999999995 (2089 tokens)
            Cache read input token costs $0.0 (0 tokens)
            Cache write input token costs $0.0 (0 tokens)
            Total cost of $0.19073400000000001
        
LANDMARK | Washington Square Park | Greenwich Village
LANDMARK | St Luke in the Fields Garden | West Village
BUSINESS | Stonewall In

In [26]:
# Again, there are generally some duplicate entities at this point, so let's convert the results into a set
unique_entities = set(entities)

In [27]:
# and fact check itself again
costs = CostTracking(model_id)
PROMPT = {}

PROMPT["SYSTEM_PROMPT"] = """
-Goal-
You are a quality assurance reviewer who accepts two inputs: a list of entities, and text which those entities 
were extracted from.  The entities come in the form of a delimited list where each line is a single entity 
and it has three categories of information: type of entity, entity name, and NYC neighborhood where it resides.
The entities are presented in the following format:
type | entity | neighborhood
Your job is to ensure each entity is mentioned in the text and not a hallucination from the original extraction.
List the entities that actually exist, one per line, in the same format:
type | entity | neighborhood

"""

PROMPT["USER_PROMPT"] = """
Entities:
{entity_list}
Text: 
{input_text}
"""

context["entity_list"] = NEWLINE.join(unique_entities)

response_text = run_llm(reference_text,context,bedrock_client,model_id,model_metadata,PROMPT,cost_tracking=costs)

# there are generally some duplicate entities at this point, so let's convert the results into a set
raw_confirmed_entities = [line for line in response_text.splitlines() if is_entity(line,3)]
confirmed_entities = set(raw_confirmed_entities)

print(costs.printVerbose())
print(response_text)

removed_entities = unique_entities.difference(set(confirmed_entities))
print(f"Confirmed entities: {NEWLINE} {NEWLINE.join(confirmed_entities)}")
print(f"Removed entities: {NEWLINE} {NEWLINE.join(removed_entities)}")


            Input token costs $0.015729 (5243 tokens)
            Output token costs $0.009614999999999999 (641 tokens)
            Cache read input token costs $0.0 (0 tokens)
            Cache write input token costs $0.0 (0 tokens)
            Total cost of $0.025344
        
Based on the text, here are the entities that are actually mentioned and confirmed:

LANDMARK | Washington Square Arch | Greenwich Village
LANDMARK | Flatiron Building | Flatiron District
BUSINESS | Stonewall Inn | West Village
BUSINESS | Pod 39 | Midtown
BUSINESS | Veselka | East Village
BUSINESS | Saks Fifth Avenue | Midtown
MUSEUM | Guggenheim Museum | Upper East Side
LANDMARK | Grace Church | East Village
LANDMARK | Union Square Park | Union Square
LANDMARK | Washington Square Park | Greenwich Village
BUSINESS | The Maiden Lane | East Village
LANDMARK | Madison Square Park | Flatiron District
LANDMARK | Rose Reading Room | Midtown
BUSINESS | Joe's Pizza | Greenwich Village
BUSINESS | Gotham Bar & Grill | U

In [28]:
# now let's compare to our single_pass results
looped_entities_improved_df = list_to_dataframe(confirmed_entities,delimiter='|',column_names=["type","name","neighborhood"]).sort_values(by="name")

improved_looped_diff_df = generate_diff_df(single_pass_entities_neighborhoods_df, looped_entities_improved_df)

print(improved_looped_diff_df)

         type_x                        name_x    neighborhood_x     type_y  \
256   LANDMARK         Empire State Building            Midtown  LANDMARK    
476   LANDMARK             Guggenheim Museum    Upper East Side    MUSEUM    
684   BUSINESS                Loeb Boathouse    Upper East Side  LANDMARK    
738   LANDMARK    Metropolitan Museum of Art    Upper East Side    MUSEUM    
1357  BUSINESS               The Plaza Hotel            Midtown  BUSINESS    

                            name_y    neighborhood_y  name_similarity_ratio  
256    American Radiator Building            Midtown               0.627451  
476             Guggenheim Museum    Upper East Side               1.000000  
684                Loeb Boathouse    Upper East Side               1.000000  
738    Metropolitan Museum of Art    Upper East Side               1.000000  
1357                  Plaza Hotel            Midtown               0.866667  


Odds are it still isn't very good at identifying the proper neighborhood consistently.  We'll address a strategy to help with that after we cover a couple other topics.

## Technique: Conversational Prompting and Prompt Caching

Up until now, we've been using an InvokeModel style API where we keep iterating over a single call by adding tokens to it.  This can be expensive however as you saw earlier, costing almost $.20 for this document. Most modern LLMs have a feature to reduce costs called prompt caching. The details of how this works is outside the scope here, but you need to know that 1/ you add a message to the prompt telling it to create a cache point, and 2/ you save a lot of money!  In the example below, which we are running in Claude Sonnet 3.7 as 3.5 does not support caching on Bedrock, you will notice that we are caching the System prompt (the instructions) and the first message, which contains the full text of the reference document (the walking tour) and is by far the majority of the tokens consumed. You'll also notice we are switching to the conversational API so we have more control over where we set these cache points.  The big difference is now we will be maintaining an array of messages instead of just a single prompt that is passed repeatedly.

In [29]:
costs = CostTracking(model_id)
model = SupportedModels.ANTHROPIC_CLAUDE_3_7_SONNET
model_id = model.value
model_metadata = get_model_metadata(model_id)
PROMPT = {}

entities = []
more_entities = True
max_iterations = 10
iterations = 0
messages = []

context["neighborhood_list"] = ",".join(neighborhood_list)

PROMPT["SYSTEM_PROMPT"] = """
-Goal-
Given the input text, extract a list of entities and determine the type of entity they are.
The valid list of entities are: 
    "BUSINESS": a location where you can purchase items or services, such as a restaurant or store.
    "MUSEUM": a location where you view historical, scientific, artistic, or cultural objects, but they are not for sale.
    "LANDMARK": a physical structure or location that is well recognized, may have historical or cultural significance, and may contain businesses, but is not a business itself. 

Classify libraries as LANDMARK.
Neighborhoods and streets as not entities.
If you know what New York City neighborhood an entity is located in, then list it in the neighborhood field. 
Use this list of neighborhoods to choose from: {neighborhood_list}
List these entities for me, one per line, in the format of:
type | entity | neighborhood

Below are a few examples of how to extract the entities.
==========
Example 1:
Text: 
Visit The Bank of New York Mellon at 240 Greenwich Street in the Tribeca neighborhood of Manhattan
Record:
BUSINESS | The Bank of New York Mellon | Tribeca
==========
Example 2:
Text: 
The Empire State Building at 20 W 34th St., New York, NY 10001 is a landmark in the Midtown neighborhood of Manhattan. Visit the Empire State Building!
Record:
LANDMARK | Empire State Building | Midtown
==========
Example 3:
Text:
The Waldorf Astoria in Midtown is a lovely hotel. I urge you to stay at The Waldorf Astoria.
Record:
BUSINESS | The Waldorf Astoria | Midtown

"""

messages.append({
    "role": "user",
    "content": [
        {
            "text": f"Text: {reference_text}"
        },
        {
            "cachePoint": {
                "type": "default"
            }
        }
    ]
})
    
while more_entities and iterations < max_iterations: 

    # Notice that we have a new parameter on this function called cache_system_prompt = True
    # This parameter adds a marking like we see in the above message for the system prompt (see line 250 of the code block)
    response = llm_conversation(
        messages, 
        context, 
        bedrock_client, 
        model_id, 
        model_metadata, 
        PROMPT, 
        cost_tracking=costs, 
        cache_system_prompt = True
    )

    output_message = response['output']['message']
    messages.append(output_message)

    new_entities = [line for line in output_message["content"][0]["text"].splitlines() if is_entity(line,3)]

    entities = entities + new_entities
    new_entity_count = len(new_entities)
    more_entities = new_entity_count > 0
    iterations = iterations + 1
    print(f"Iteration {iterations}. Found {new_entity_count} new entities. {len(entities)} entities total now")
    
    if more_entities:
        content = [
            {"text": """Are you sure you got all of the entities?  Please check again and add any entities in the same format
            as before. Only include new entities you found. I can see the history of what you told me already."""}
        ]
        if iterations in [2,4,6]:
            content.append(
                {
                    "cachePoint": {
                        "type": "default"
                    }
                }
            )

        messages.append({
            "role": "user",
            "content": content
        })
    
print("==== FULL CONVERSATION ====")
print(messages)
print("==== END CONVERSATION ====")
print(costs.printVerbose())


Iteration 1. Found 43 new entities. 43 entities total now
Iteration 2. Found 9 new entities. 52 entities total now
Iteration 3. Found 1 new entities. 53 entities total now
Iteration 4. Found 0 new entities. 53 entities total now
==== FULL CONVERSATION ====
[{'role': 'user', 'content': [{'text': 'Text: \nThe walking tour route I am sharing here takes you from downtown Manhattan, through mid-town to Central Park and the Upper East Side, taking in many NYC icons along the way, including:\n\nGreenwich Village & Washington Square Park\nFlatiron Building & Madison Square Park\nEmpire State Buiding\nNew York Public Library\nFifth Avenue Shopping\nRockefeller Centre & Top Of The Rock\nPlaza Hotel & Pulitzer Fountain\nCentral Park\n\nThere are other routes that I like in Manhattan and other neighbourhoods which are great to explore – but I wanted to start with something that was doable in one day, and which covers several of the major landmarks in New York. I walked this route on a visit to New

In this query, we implemented the extraction as a conversation instead of a loop. Second, we took advantage of a newer feature in LLM APIs called "prompt caching", which stores the previous parts of a conversation instead of rerunning them through the LLM, providing cheaper prices.  Here we cached the original system prompt, the full original text, and then the outputs of every 2 iterations.  You can see that the overall cost of this extraction was probably between \\$0.01 and \\$0.02 (it will vary based on the number of entities extracted and number of iterations the LLM decides to do). As a comparison, let's rerun our loop approach again but with Claude 3.7 so we can do an apples to apples comparision (again, it won't be exact because every run will vary in number of iterations and the exact number of tokens returned in each iteration).

In [30]:
costs = CostTracking(model_id)
PROMPT = {}

entities = []
more_entities = True
max_iterations = 10
iterations = 0

while more_entities and iterations < max_iterations: 
    PROMPT["SYSTEM_PROMPT"] = """
    -Goal-
    Given the input text, extract a list of entities and determine the type of entity they are.
    The valid list of entities are: 
        "BUSINESS": a location where you can purchase items or services, such as a restaurant or store.
        "MUSEUM": a location where you view historical, scientific, artistic, or cultural objects, but they are not for sale.
        "LANDMARK": a physical structure or location that is well recognized, may have historical or cultural significance, and may contain businesses, but is not a business itself. 

    Classify libraries as LANDMARK.
    Neighborhoods and streets as not entities.
    If you know what New York City neighborhood an entity is located in, then list it in the neighborhood field. 
    Use this list of neighborhoods to choose from: {neighborhood_list}
    List these entities for me, one per line, in the format of:
    type | entity | neighborhood

    Below are a few examples of how to extract the entities.
    ==========
    Example 1:
    Text: 
    Visit The Bank of New York Mellon at 240 Greenwich Street in the Tribeca neighborhood of Manhattan
    Record:
    BUSINESS | The Bank of New York Mellon | Tribeca
    ==========
    Example 2:
    Text: 
    The Empire State Building at 20 W 34th St., New York, NY 10001 is a landmark in the Midtown neighborhood of Manhattan. Visit the Empire State Building!
    Record:
    LANDMARK | Empire State Building | Midtown
    ==========
    Example 3:
    Text:
    The Waldorf Astoria in Midtown is a lovely hotel. I urge you to stay at The Waldorf Astoria.
    Record:
    BUSINESS | The Waldorf Astoria | Midtown

    There may be entities you already identified in the list below:
    {entities}
    Please check this text for entities that were not already identified, and please list them if there are.
    """

    PROMPT["USER_PROMPT"] = """
    Text: {input_text}
    """

    context["entities"] = NEWLINE.join(entities)

    response_text = run_llm(reference_text,context,bedrock_client,model_id,model_metadata,PROMPT,cost_tracking=costs,print_prompts=False)

    new_entities = [line for line in response_text.splitlines() if is_entity(line,3)]

    entities = entities + new_entities
    new_entity_count = len(new_entities)
    more_entities = new_entity_count > 0
    iterations = iterations + 1
    print(f"Iteration {iterations}. Found {new_entity_count} new entities. {len(entities)} entities total now")

print(NEWLINE.join(entities))
print(costs.printVerbose())


Iteration 1. Found 44 new entities. 44 entities total now
Iteration 2. Found 7 new entities. 51 entities total now
Iteration 3. Found 16 new entities. 67 entities total now
Iteration 4. Found 7 new entities. 74 entities total now
Iteration 5. Found 7 new entities. 81 entities total now
I'm being throttled.  Sleeping for 1 second.
Iteration 6. Found 19 new entities. 100 entities total now
I'm being throttled.  Sleeping for 1 second.
Iteration 7. Found 28 new entities. 128 entities total now
I'm being throttled.  Sleeping for 1 second.
Iteration 8. Found 25 new entities. 153 entities total now
I'm being throttled.  Sleeping for 1 second.
Iteration 9. Found 40 new entities. 193 entities total now
Iteration 10. Found 8 new entities. 201 entities total now
LANDMARK | Washington Square Park | Greenwich Village
LANDMARK | Flatiron Building | Flatiron District
LANDMARK | Empire State Building | Midtown
LANDMARK | New York Public Library | Midtown
LANDMARK | Rockefeller Centre | Midtown
LANDMAR

The exact cost of this nature of request isn't deterministic, but you should see that where our conversational API had a majority of the tokens consumed as "Cache read input tokens" which likely cost a fraction of a penny, this looping approach likely had a majority consumed as "Input tokens" which likely cost well over \\$0.10 if not \\$0.20.

# Strategy: Prompt Chaining

Now we are going to explore chaining various prompts together to get more focused and uniform results.  We will also be working on a couple other concepts in this section: relationship extraction and tool usage.

In our first step, we are going to modify our LLM prompt to extract the neighborhoods discussed in the walking tour, as well as the suggested order to visit the neighborhoods.  The latter will be our relationships between the neighborhoods.  One more change we'll make is to ask the LLM to output the nodes and edges in a CSV format we can use to load data into our graph database as well as a list format.

In [31]:
costs = CostTracking(model_id)
PROMPT = {}
PROMPT["SYSTEM_PROMPT"] = """
You will be provided an article of a walking tour of New York City. Please go through the article and 
create a list of New York City neighborhoods that are mentioned in the article.  Provide the list of neighborhoods, 
one per line. There should be a new line after the last neighborhood and before the closing tag.

Then you will provide those exact same neighborhoods in a CSV format within the <nodes> tag.  
The first line is the header exactly as shown in the example.
Then for each neighborhood, add a line that contains: 
- the neighborhood name should be in snake case prefixed with "neighborhood_" and stored as :ID
- the :LABEL must be NEIGHBORHOOD
- the name:String field will be the neighborhood name
- the is_start:Bool field should be true if the article suggests starting the tour there

Also track the order of visiting each neighborhood as suggested by the text.
The order will be provided in the <edges> tag as a CSV formatted list
The first line is the header exactly as shown in the example.
Add at least one line for every neighborhood listed in nodes which has a suggested neighborhood to visit afterwards in this list using the following field definitions:
- :ID is a RFC 4122 compliant GUID
- :START_ID is the current neighborhood :ID listed in snake case exactly as shown in <nodes>
- :END_ID is the next neighborhood :ID listed in snake case exactly as shown in <nodes>
- :TYPE is always NEXT_NEIGHBORHOOD

Always ensure you have both the opening and closing tags for both neighborhoods, nodes and, edges

<neighborhoods>
neighborhood
neighborhood
neighborhood
</neighborhoods>
<nodes>
:ID, :LABEL, name:String, is_start:Bool
neighborhood_name_camel_case, NEIGHBORHOOD, neighborhood name, true
neighborhood_name_2_camel_case, NEIGHBORHOOD, neighborhood name 2, false
</nodes>
<edges>
:ID, :START_ID, :END_ID, :TYPE
guid, neighborhood_name_camel_case, neighborhood_name_2_camel_case, NEXT_NEIGHBORHOOD
guid, neighborhood_name_2_camel_case, neighborhood_name_3_camel_case, NEXT_NEIGHBORHOOD
</edges>


Do not provide any other explanatory text. Ensure you have captured all of the details from the text in your response.  
"""

PROMPT["USER_PROMPT"] = """
Text: {input_text}
"""
    
response_text = run_llm(reference_text,context,bedrock_client,model_id,model_metadata,PROMPT,cost_tracking=costs)

lines = response_text.splitlines(keepends=True)
saved_neighborhoods = []
nodes = []
edges = []

in_neighborhoods_tag = False
in_nodes_tag = False
in_edges_tag = False
for line in lines:
    match line.strip(): 
        case "<neighborhoods>": 
            in_neighborhoods_tag = True
            in_nodes_tag = False
            in_edges_tag = False
        case "</neighborhoods>": 
            in_neighborhoods_tag = False
        case "<nodes>": 
            in_neighborhoods_tag = False
            in_nodes_tag = True
            in_edges_tag = False
        case "</nodes>": 
            in_nodes_tag = False
        case "<edges>": 
            in_neighborhoods_tag = False
            in_nodes_tag = False
            in_edges_tag = True
        case "</edges>": 
            in_edges_tag = False
        case _: 
            if in_neighborhoods_tag:
                saved_neighborhoods.append(line.strip())
            elif in_nodes_tag:
                nodes.append(line.strip())
            elif in_edges_tag:
                edges.append(line.strip())

print(saved_neighborhoods)

print(costs.printVerbose())

I'm being throttled.  Sleeping for 1 second.
['Greenwich Village', 'Washington Square Park', 'Union Square Park', 'Flatiron Building & Madison Square Park', 'Empire State Building', 'New York Public Library', 'Fifth Avenue', 'Rockefeller Centre', 'Plaza Hotel & Pulitzer Fountain', 'Central Park', 'Upper East Side', 'East Village']

            Input token costs $0.014379000000000001 (4793 tokens)
            Output token costs $0.012674999999999999 (845 tokens)
            Cache read input token costs $0.0 (0 tokens)
            Cache write input token costs $0.0 (0 tokens)
            Total cost of $0.027054
        


<div class="alert alert-block alert-info">You probably also noticed that some of these neighborhoods it extracted are a little bit suspect (you might see "Empire State Building", "Rockefeller Center", "Flatiron Building"...these aren't neighborhoods). 
We could apply our earlier technique of constraining the choices, but let's go without it for now.</div>

Before we get too far invested in this strategy, let's look at the results for just the first neighborhood, which usually is Greenwich Village.  If you didn't get Greenwich Village, run the previous cell again until you do.

In [32]:
costs = CostTracking(model_id)

PROMPT = {}
PROMPT["SYSTEM_PROMPT"] = """
-Goal-
Given the reference text and a New York City neighborhood, extract a list of entities within that neighborhood and determine the type of entity they are.
The valid list of entity_types are: 
    "BUSINESS": a location where you can purchase items or services, such as a restaurant or store.
    "MUSEUM": a location where you view historical, scientific, artistic, or cultural objects, but they are not for sale.
    "LANDMARK": a physical structure or location that is well recognized, may have historical or cultural significance, and may contain businesses, but is not a business itself. 

Classify libraries as LANDMARK.
Neighborhoods and streets as not entities.
Assign each entity a sequential ID value
If the entity_type is a BUSINESS, assign it one of these entity_subtype. If you are unsure, propose a new entity_subtype in that field:
RESTAURANT
STORE
HOTEL

If the entity_type is MUSEUM, always assign MUSEUM to the entity_subtype
If the entity_type is a LANDMARK, assign it one of these entity_subtype. If you are unsure, propose a new entity_subtype in that field:
PARK
LIBRARY
BUILDING

List these entities for me, one per line, in the format of:
<entities>
ID | entity_type | entity_name | entity_subtype
</entities>
Always ensure you have both the opening and closing tags for entities
"""

messages = []

neighborhood_name = 'Greenwich Village'
print(f"Starting neighborhood '{neighborhood_name}':")
neighborhood_id = 'greenwich_village'
context["neighborhood_name"] = neighborhood_name
PROMPT["USER_PROMPT"] = """
Neighborhood Name: {neighborhood}
Reference Text: {input_text}
"""

initial_message = {
    "role": "user",
    "content": [
        {
            "text": f"""
                        Neighborhood Name: {neighborhood_name}
                        Reference Text: {reference_text}
                    """
        },
        {
            "cachePoint": {
                "type": "default"
            }
        }
    ]
}

messages.append(initial_message)

response = llm_conversation(
    messages, 
    context, 
    bedrock_client, 
    model_id, 
    model_metadata, 
    PROMPT, 
    cost_tracking=costs, 
    cache_system_prompt = True
)

output_message = response['output']['message']
print(output_message["content"][0]["text"])


Starting neighborhood 'Greenwich Village':
I'll extract the entities within Greenwich Village from the reference text and determine their types.

<entities>
1 | BUSINESS | St Luke in the Fields Garden | PARK
2 | BUSINESS | Friends building | BUILDING
3 | LANDMARK | Christopher Park | PARK
4 | BUSINESS | Stonewall Inn | RESTAURANT
5 | BUSINESS | Joe's Pizza | RESTAURANT
6 | LANDMARK | Jefferson Market Library | LIBRARY
7 | BUSINESS | Carrie Bradshaw's apartment | BUILDING
8 | BUSINESS | Angelika Theatre | STORE
9 | LANDMARK | Washington Square Park | PARK
10 | LANDMARK | Washington Square Arch | BUILDING
11 | LANDMARK | One Fifth Avenue | BUILDING
12 | BUSINESS | Gotham Bar & Grill | RESTAURANT
</entities>


You probably see some that aren't in Greenwich Village, like the Empire State Building or Rockefeller Center. Let's explore using a location service to help the LLM with this.

# Strategy: Tool Usage

Remember that LLMs are, at their core, a text generation tool trained at a moment in time. Tools (or functions) are an extension of LLM's capabilities to enable it to perform more complex tasks.
We provide the LLM a set of tools and a description of what they do, and the LLM decides whether to use them and how to utilize them.
In this case, we are going to provide our LLM a tool called "Location_Tool". It allows the LLM to pass in an entity name, and which neighborhood it thinks that entity exists in. The tool then calls the AWS Location Geoplaces service to find the closest match and returns the name and address, as well as the neighborhood that is the closest match.  The tool also checks the service's confidence of the top result and if it is below a threshold, instead it tells the LLM that it couldn't find a high confidence result.
Keep in mind that the tool is running fully in the context of our AWS account. We aren't somehow giving the LLM the ability to connect to our instance of the Location service. Instead it works as such:
- We prompt the LLM and tell it about the tool it has available to use if it wants to.
- The LLM response is coded as "tool_usage" if it wants to call the tool, and it includes the parameters we defined in the request.
- We handle the response and look for that code.  If it is "tool_usage" then we call the proper tool (there may be many) with the provided parameters.
- When we get the response from the tool, we add a new message to the conversation with the tool's output.
- This cycle continues until the LLM gives us a message coded "end_turn".  That tells us the message body is the LLMs answer to the prompt.

To accommodate this functionality, we are adding 2 new functions below as well as a tool specification.<br>
The tool specification tells the LLM the name of the tool (which we reference in the prompt), what it does, and how to call it.<br>
The "fetch_location_data" function is the function that calls the Location service itself.  E.g., it is the functional component of the tool.
The "process_response" function inspects the LLM's message and either executes the tool request or tells our control loop to continue with the response.

NOTE: You may be asking why we are using tool usage instead of Model Context Protocol (MCP) here. MCP requires setting up a server and client, adding complexity to an already complex workshop. At its core, MCP utilizes the same API calls as tool usage. If you'd like to learn more about using MCP with Amazon Bedrock, I recommend this [article from community.aws](https://community.aws/content/2uFvyCPQt7KcMxD9ldsJyjZM1Wp/model-context-protocol-mcp-and-amazon-bedrock). 


In [33]:
# This describes to Bedrock and the FM the tool that is available and how to use it.
tool_spec = {
    "toolSpec": {
        "name": "Location_Tool",
            "description": "Get the full address and neighborhood for named location in Manhattan",
            "inputSchema": {
                "json": {
                    "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "The name of the location.",
                    },
                    "assumed_neighborhood": {
                        "type": "string",
                        "description": "The Manhattan neighborhood where I think it is located.",
                    },
                },
                "required": ["name", "assumed_neighborhood"],
                }
            },
        }
    }
    
# This is the actual function that will be executed when the FM uses the tool.
geoplaces_client = boto3.client('geo-places')

def fetch_location_data(input_data):
    name = input_data.get("name")
    assumed_neighborhood = input_data.get("assumed_neighborhood")
    print(f"Location Data Request: name={name},neighborhood={assumed_neighborhood}")

    try:
        response = geoplaces_client.geocode(
            QueryText=name,
            QueryComponents={
              "Country":"United States",
              "Region": "New York",
              "Locality": "Manhattan",
              "District": assumed_neighborhood
            }
        )
#        print(f"API Response: {response}")
        
        if response["ResultItems"][0]['MatchScores']['Overall'] < .75:
            return {"error": "true", "message": "A high confidence match cannot be found for this location."}
        else:
            return {
                "Label": response["ResultItems"][0]["Address"]["Label"],
                "Neighborhood": response["ResultItems"][0]["Address"]["District"]
            }
    except Exception as e:
        return {"error": type(e), "message": str(e)}
        
# This inspects the last message from the FM and determines the proper action based on its contents.
def process_response(conversation_response):
    continue_conversation = True
    tool_results = []
    match conversation_response["stopReason"]:
        case "tool_use":
            for content_block in conversation_response["output"]["message"]["content"]:
                if "text" in content_block:
                    print(f"===LLM Commentary==={NEWLINE}{content_block['text']}{NEWLINE}===End LLM Commentary===")
                elif "toolUse" in content_block:
                    match content_block["toolUse"]["name"]:
                        case "Location_Tool":
                            print(f"Calling Location_Tool for input {content_block['toolUse']['input']}")
                            answer = fetch_location_data(content_block["toolUse"]["input"])
                            item_response = answer
                        case _:
                            error_message = (
                                f"The requested tool with name '{content_block['toolUse']['name']}' does not exist."
                            )
                            item_response = {"error": "true", "message": error_message}
                    tool_results.append(
                        {
                            "toolResult": {
                                "toolUseId": content_block["toolUse"]["toolUseId"],
                                "content": [{"json": item_response}]
                            }
                        }
                    )
                    print(f"Location_Tool response is {item_response}")

        case "end_turn":
            for content_block in conversation_response["output"]["message"]["content"]:
                if "text" in content_block:
                    tool_results.append(content_block)
            continue_conversation = False
            
        case _:
            continue_conversation = False
            tool_results.append({"error": "true", "message": f"Unknown stopReason {conversation_response['stopReason']}"})

    return {
        "continueConversation": continue_conversation,
        "response": tool_results
    }

Now let's try calling that same prompt we did before for Greenwich Village, but this time we are going to use our new Location_Tool. So we need to make a couple changes:
1. We add this section of instructions to the system prompt
<pre>If you aren't sure whether an entity is located within the proper neighborhood, use the Location_Tool to verify it is in the neighborhood.
You will provide Location_Tool with the name of the entity and the neighborhood you believe it is in. 
It will return the Label field with the full address of name and address of the entity, and the Neighborhood field with the neighborhood the tool believes where it exists.
If you agree that the entity is in the neighborhood being searched, then include it there.</pre>
2. We pass the tool specification into the Converse API call.

For this first run, we are going to step through the process instead of running it in a single cell.  If you don't get a desired output here, make sure to return to this cell first.  If you just rerun a single cell after this point, you will likely get an error because remember our conversations are cumulative in the Converse API.

In [34]:
costs = CostTracking(model_id)

PROMPT = {}
PROMPT["SYSTEM_PROMPT"] = """
-Goal-
Given the reference text and a New York City neighborhood, extract a list of entities within that neighborhood and determine the type of entity they are.
The valid list of entity_types are: 
    "BUSINESS": a location where you can purchase items or services, such as a restaurant or store.
    "MUSEUM": a location where you view historical, scientific, artistic, or cultural objects, but they are not for sale.
    "LANDMARK": a physical structure or location that is well recognized, may have historical or cultural significance, and may contain businesses, but is not a business itself. 

Classify libraries as LANDMARK.
Neighborhoods and streets as not entities.
Assign each entity a sequential ID value
If the entity_type is a BUSINESS, assign it one of these entity_subtype. If you are unsure, propose a new entity_subtype in that field:
RESTAURANT
STORE
HOTEL

If the entity_type is MUSEUM, always assign MUSEUM to the entity_subtype
If the entity_type is a LANDMARK, assign it one of these entity_subtype. If you are unsure, propose a new entity_subtype in that field:
PARK
LIBRARY
BUILDING

If you aren't sure whether an entity is located within the proper neighborhood, use the Location_Tool to verify it is in the neighborhood.
You will provide Location_Tool with the name of the entity and the neighborhood you believe it is in. 
It will return the Label field with the full address of name and address of the entity, and the Neighborhood field with the neighborhood the tool believes where it exists.
If you agree that the entity is in the neighborhood being searched, then include it there.

List these entities for me, one per line, followed by the visit order in the format of:
<entities>
ID | entity_type | entity_name | entity_subtype
</entities>
Always ensure you have both the opening and closing tags for entities
"""

messages = []

neighborhood_name = 'Greenwich Village'
print(f"Starting neighborhood '{neighborhood_name}':")
neighborhood_id = 'greenwich_village'
context["neighborhood_name"] = neighborhood_name
PROMPT["USER_PROMPT"] = """
Neighborhood Name: {neighborhood}
Reference Text: {input_text}
"""

initial_message = {
    "role": "user",
    "content": [
        {
            "text": f"""
                        Neighborhood Name: {neighborhood_name}
                        Reference Text: {reference_text}
                    """
        },
        {
            "cachePoint": {
                "type": "default"
            }
        }
    ]
}

messages.append(initial_message)

initial_llm_response = llm_conversation(
    messages, 
    context, 
    bedrock_client, 
    model_id, 
    model_metadata, 
    PROMPT, 
    cost_tracking=costs, 
    cache_system_prompt = True,
    tools = [tool_spec]
)

print(initial_llm_response["output"]["message"])
messages.append(initial_llm_response["output"]["message"])

Starting neighborhood 'Greenwich Village':
{'role': 'assistant', 'content': [{'text': "I'll help you extract and categorize entities in Greenwich Village based on the reference text. I'll need to identify businesses, museums, and landmarks found in the Greenwich Village neighborhood.\n\nFirst, let me identify the entities mentioned in Greenwich Village from the reference text:"}, {'toolUse': {'toolUseId': 'tooluse_Y9Sr2MS1T1Sc5xEOBOC28g', 'name': 'Location_Tool', 'input': {'name': 'St Luke in the Fields Garden', 'assumed_neighborhood': 'Greenwich Village'}}}]}


Next we take the LLMs response and run it through our function. We also print out any commentary the LLM returns about how/why it is calling the tool. Almost always we will see the LLM utilizing the tool here and we assume it will, but if not then just rerun the prior step.
Our last line takes the response from the tool and appends it to the list of messages.

In [35]:
tool_response = process_response(initial_llm_response)
should_continue_conversation = tool_response["continueConversation"]
messages.append({"role": "user", "content": tool_response["response"]})

===LLM Commentary===
I'll help you extract and categorize entities in Greenwich Village based on the reference text. I'll need to identify businesses, museums, and landmarks found in the Greenwich Village neighborhood.

First, let me identify the entities mentioned in Greenwich Village from the reference text:
===End LLM Commentary===
Calling Location_Tool for input {'name': 'St Luke in the Fields Garden', 'assumed_neighborhood': 'Greenwich Village'}
Location Data Request: name=St Luke in the Fields Garden,neighborhood=Greenwich Village
Location_Tool response is {'error': 'true', 'message': 'A high confidence match cannot be found for this location.'}


Now we call the LLM again with the same prompts and the message list from before except with the tool response added to it. Again, we always make sure to append the LLMs response to our message list also. It doesn't add it on its own.

In [36]:
llm_response = llm_conversation(
        messages, 
        context, 
        bedrock_client, 
        model_id, 
        model_metadata, 
        PROMPT, 
        cost_tracking=costs, 
        cache_system_prompt = True,
        tools = [tool_spec]
    )
print(llm_response["output"]["message"])
messages.append(llm_response["output"]["message"])

{'role': 'assistant', 'content': [{'text': 'Let me try to verify some more locations:'}, {'toolUse': {'toolUseId': 'tooluse_h5f_s5c3TTmc0-onu_FA3A', 'name': 'Location_Tool', 'input': {'name': 'Washington Square Park', 'assumed_neighborhood': 'Greenwich Village'}}}]}


And again, we process the response.  Here we are going to set up a processing loop to let the LLM call the tool up to 20 more times (we don't want a runaway LLM running up a big bill!).  Eventually it will get all it needs from the tool and output its answer.  I think you'll see if you have a much more accurate list of entities that are really in Greenwich Village.  The commentary of the output will be interesting also, as you'll likely see how the LLM reasons that even if the tool says something is in the West Village neighborhood, that neighborhood is a subset of Greenwich Village.

In [37]:
tool_response = process_response(llm_response)
should_continue_conversation = tool_response["continueConversation"]
messages.append({"role": "user", "content": tool_response["response"]})

iteration = 0
max_iterations = 20

while should_continue_conversation and iteration < max_iterations:
    iteration = iteration + 1
    print(f"conversation iteration {iteration} of up to {max_iterations}")

    llm_response = llm_conversation(
        messages, 
        context, 
        bedrock_client, 
        model_id, 
        model_metadata, 
        PROMPT, 
        cost_tracking=costs, 
        cache_system_prompt = True,
        tools = [tool_spec]
    )
    messages.append(llm_response["output"]["message"])
    
    tool_response = process_response(llm_response)
    should_continue_conversation = tool_response["continueConversation"]
    if should_continue_conversation:
        messages.append({"role": "user", "content": tool_response["response"]})
    print(tool_response)
        
    print(f"are we continuing the conversation? {should_continue_conversation}")

for item in tool_response["response"]:
    if "text" in item:
        print(item["text"])
    else:
        print(item)
                    

===LLM Commentary===
Let me try to verify some more locations:
===End LLM Commentary===
Calling Location_Tool for input {'name': 'Washington Square Park', 'assumed_neighborhood': 'Greenwich Village'}
Location Data Request: name=Washington Square Park,neighborhood=Greenwich Village
Location_Tool response is {'Label': 'Washington Square Park, 2 5th Ave, New York, NY 10011-0034, United States', 'Neighborhood': 'Greenwich Village'}
conversation iteration 1 of up to 20
Calling Location_Tool for input {'name': 'Stonewall Inn', 'assumed_neighborhood': 'Greenwich Village'}
Location Data Request: name=Stonewall Inn,neighborhood=Greenwich Village
Location_Tool response is {'Label': 'The Stonewall Inn, 53 Christopher St, New York, NY 10014-3530, United States', 'Neighborhood': 'West Village'}
{'continueConversation': True, 'response': [{'toolResult': {'toolUseId': 'tooluse_pXP3m19JRoWi9D2y7B1amA', 'content': [{'json': {'Label': 'The Stonewall Inn, 53 Christopher St, New York, NY 10014-3530, Unite

# Extracting Relationships and Pulling It All Together

Now we are going to leverage many of the strategies we learned for using an LLM for entity extraction to build an actual knowledge graph.
We are going to instruct the LLM to output the data in a way that we can easily load it into an Amazon Neptune instance running in this account.

These functions should still be defined, but I've included them here as a refresher and so you can more easily follow along.

In [38]:
# This describes to Bedrock and the FM the tool that is available and how to use it.
tool_spec = {
    "toolSpec": {
        "name": "Location_Tool",
            "description": "Get the full address and neighborhood for named location in Manhattan",
            "inputSchema": {
                "json": {
                    "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "The name of the location.",
                    },
                    "assumed_neighborhood": {
                        "type": "string",
                        "description": "The Manhattan neighborhood where I think it is located.",
                    },
                },
                "required": ["name", "assumed_neighborhood"],
                }
            },
        }
    }
    
# This is the actual function that will be executed when the FM uses the tool.
geoplaces_client = boto3.client('geo-places')

def fetch_location_data(input_data):
    name = input_data.get("name")
    assumed_neighborhood = input_data.get("assumed_neighborhood")

    try:
        response = geoplaces_client.geocode(
            QueryText=name,
            QueryComponents={
              "Country":"United States",
              "Region": "New York",
              "Locality": "Manhattan",
              "District": assumed_neighborhood
            }
        )
        
        if response["ResultItems"][0]['MatchScores']['Overall'] < .75:
            print(f"The top location only had a certainty score of {response['ResultItems'][0]['MatchScores']['Overall']} so returning no result.")
            return {"error": "true", "message": "A high confidence match cannot be found for this location."}
        else:
            print(f"Top match by the location service was label={response['ResultItems'][0]['Address']['Label']}; neighborhood={response['ResultItems'][0]['Address']['District']})")
            return {
                "Label": response["ResultItems"][0]["Address"]["Label"],
                "Neighborhood": response["ResultItems"][0]["Address"]["District"]
            }
    except Exception as e:
        return {"error": type(e), "message": str(e)}
        
# This inspects the last message from the FM and determines the proper action based on its contents.
def process_response(conversation_response):
    continue_conversation = True
    tool_results = []
    match conversation_response["stopReason"]:
        case "tool_use":
            for content_block in conversation_response["output"]["message"]["content"]:
                if "text" in content_block:
                    print(f"===LLM Commentary==={NEWLINE}{content_block['text']}{NEWLINE}===End LLM Commentary===")
                elif "toolUse" in content_block:
                    match content_block["toolUse"]["name"]:
                        case "Location_Tool":
                            print(f"Calling Location_Tool for input {content_block['toolUse']['input']}")
                            answer = fetch_location_data(content_block["toolUse"]["input"])
                            item_response = answer
                        case _:
                            error_message = (
                                f"The requested tool with name '{content_block['toolUse']['name']}' does not exist."
                            )
                            item_response = {"error": "true", "message": error_message}
                    tool_results.append(
                        {
                            "toolResult": {
                                "toolUseId": content_block["toolUse"]["toolUseId"],
                                "content": [{"json": item_response}]
                            }
                        }
                    )

        case "end_turn":
            for content_block in conversation_response["output"]["message"]["content"]:
                if "text" in content_block:
                    tool_results.append(content_block)
            continue_conversation = False
            
        case _:
            continue_conversation = False
            tool_results.append({"error": "true", "message": f"Unknown stopReason {conversation_response['stopReason']}"})

    return {
        "continueConversation": continue_conversation,
        "response": tool_results
    }

Remembering our prompt chaining strategy, first, we will get a list of neighborhoods instead of asking the LLM to focus on everything at once. We also included instructions so it prints the results in the CSV format we desire.

In [39]:
costs = CostTracking(model_id)

NEIGHBORHOOD_PROMPT = {}
NEIGHBORHOOD_PROMPT["SYSTEM_PROMPT"] = """
You will be provided an article of a walking tour of New York City. Please go through the article and 
create a list of New York City neighborhoods that are mentioned in the article.  Provide the list of neighborhoods, 
one per line. There should be a new line after the last neighborhood and before the closing tag.

Then you will provide those exact same neighborhoods in a CSV format within the <nodes> tag.  
The first line is the header exactly as shown in the example.
Then for each neighborhood, add a line that contains: 
- the neighborhood name should be in snake case prefixed with "neighborhood_" and stored as :ID
- the :LABEL must be NEIGHBORHOOD
- the name:String field will be the neighborhood name
- the is_start:Bool field should be true if the article suggests starting the tour there

Also track the order of visiting each neighborhood as suggested by the text.
The order will be provided in the <edges> tag as a CSV formatted list
The first line is the header exactly as shown in the example.
Add at least one line for every neighborhood listed in nodes which has a suggested neighborhood to visit afterwards in this list using the following field definitions:
- :ID is a RFC 4122 compliant GUID
- :START_ID is the current neighborhood :ID listed in snake case exactly as shown in <nodes>
- :END_ID is the next neighborhood :ID listed in snake case exactly as shown in <nodes>
- :TYPE is always NEXT_NEIGHBORHOOD

Always ensure you have both the opening and closing tags for both nodes and edges

<nodes>
:ID, :LABEL, name:String, is_start:Bool
neighborhood_name_camel_case, NEIGHBORHOOD, neighborhood name, true
neighborhood_name_2_camel_case, NEIGHBORHOOD, neighborhood name 2, false
</nodes>
<edges>
:ID, :START_ID, :END_ID, :TYPE
guid, neighborhood_name_camel_case, neighborhood_name_2_camel_case, NEXT_NEIGHBORHOOD
guid, neighborhood_name_2_camel_case, neighborhood_name_3_camel_case, NEXT_NEIGHBORHOOD
</edges>


Do not provide any other explanatory text. Ensure you have captured all of the details from the text in your response.  
"""

NEIGHBORHOOD_PROMPT["USER_PROMPT"] = """
Text: {input_text}
"""

# This parameter instructs the FM it can return up to 8192 tokens in its response.
context["max_tokens"] = 8192

response_text = run_llm(reference_text,context,bedrock_client,model_id,model_metadata,NEIGHBORHOOD_PROMPT,cost_tracking=costs)

lines = response_text.splitlines(keepends=True)
neighborhood_nodes = []
neighborhood_edges = []

in_nodes_tag = False
in_edges_tag = False
for line in lines:
    match line.strip(): 
        case "<nodes>": 
            in_nodes_tag = True
            in_edges_tag = False
        case "</nodes>": 
            in_nodes_tag = False
        case "<edges>": 
            in_nodes_tag = False
            in_edges_tag = True
        case "</edges>": 
            in_edges_tag = False
        case _: 
            if in_nodes_tag:
                neighborhood_nodes.append(line.strip())
            elif in_edges_tag:
                neighborhood_edges.append(line.strip())
                
print("The neighborhoods that were identified are:")
for line in neighborhood_nodes[1:]:
    fields = line.split(',')
    print(f"{fields[0]}:{fields[2]}")


The neighborhoods that were identified are:
neighborhood_greenwich_village: Greenwich Village
neighborhood_washington_square_park: Washington Square Park
neighborhood_union_square_park: Union Square Park
neighborhood_flatiron_building: Flatiron Building
neighborhood_madison_square_park: Madison Square Park
neighborhood_empire_state_building: Empire State Building
neighborhood_midtown: Midtown
neighborhood_new_york_public_library: New York Public Library
neighborhood_bryant_park: Bryant Park
neighborhood_rockefeller_centre: Rockefeller Centre
neighborhood_central_park: Central Park
neighborhood_upper_east_side: Upper East Side


NOTE:  It is very possible that the LLM went off the rails and started hallucinating neighborhoods like Empire State Building, New York Public Library, Rockefeller Centre, The Plaza Hotel, or Pulitzer Fountain.  You can rerun the previous step if you want or just let the LLM run wild with it and see what happens.  Spoiler Alert: It is interesting to see how the LLM may somewhat course correct with some of these neighborhoods later as it collects location data from the tool.<br>
Pro Tip: In a production application, you likely want to provide the LLM an actual list of NYC neighborhoods or a tool it can use to verify them. But this is more interesting! <br><br>
We are going to define two sets of prompts:
- The first prompt is the one we used above to extract and verify the entities, by neighborhood, using the Location_Tool as needed.</li>
- The second prompt:
- - takes the list of entities from the first prompt and formats it as a CSV file we can load into Amazon Neptune
- - determines the proper visit order of each stop within the neighborhood and generates a relationship between those two nodes
- - determines the last stop within the neighborhood and adds a relationship to the neighborhood itself signifying it is time to exit the neighborhood
- - outputs all of the relationships in a CSV format we can load into Amazon Neptune

We have a set of dictionaries for storing the nodes and edges by neighborhood for now.  Later we will write them to disk as CSV files and then load them to Neptune.

This cell is going to take a while to run.  We've included some verbose output as it runs so you can follow what it is doing in real-time.

In [40]:
# now we are going to run through the conversational model with tool for each neighborhood.
# First we will get the list of entities and later we will extract the edges.
nodes_by_neighborhood = {}
edges_by_neighborhood = {}

ENTITY_PROMPT_1 = {}
ENTITY_PROMPT_1["SYSTEM_PROMPT"] = """
-Goal-
Given the reference text and a New York City neighborhood, extract a list of entities within that neighborhood and determine the type of entity they are.
The valid list of entity_types are: 
    "BUSINESS": a location where you can purchase items or services, such as a restaurant or store.
    "MUSEUM": a location where you view historical, scientific, artistic, or cultural objects, but they are not for sale.
    "LANDMARK": a physical structure or location that is well recognized, may have historical or cultural significance, and may contain businesses, but is not a business itself. 

Classify libraries as LANDMARK.
Neighborhoods and streets as not entities.
Assign each entity a sequential ID value
If the entity_type is a BUSINESS, assign it one of these entity_subtype. If you are unsure, propose a new entity_subtype in that field:
RESTAURANT
STORE
HOTEL

If the entity_type is MUSEUM, always assign MUSEUM to the entity_subtype
If the entity_type is a LANDMARK, assign it one of these entity_subtype. If you are unsure, propose a new entity_subtype in that field:
PARK
LIBRARY
BUILDING

If you aren't sure whether an entity is located within the proper neighborhood, use the Location_Tool to verify it is in the neighborhood.
You will provide Location_Tool with the name of the entity and the neighborhood you believe it is in. 
It will return the Label field with the full address of name and address of the entity, and the Neighborhood field with the neighborhood the tool believes where it exists.
If you agree that the entity is in the neighborhood being searched, then include it there.

List these entities for me, one per line, followed by the visit order in the format of:
<entities>
ID | entity_type | entity_name | entity_subtype
</entities>
Always ensure you have both the opening and closing tags for entities
"""

ENTITY_PROMPT_1["USER_PROMPT"] = """
Neighborhood Name: {neighborhood_name}
Reference Text: {input_text}
"""

ENTITY_PROMPT_2 = {}
ENTITY_PROMPT_2["SYSTEM_PROMPT"] = """
-Goal-
You are provided a reference text representing a walking tour of New York City, a list of entities mentioned in the reference text, 
a neighborhood_name you are focusing on, and a neighborhood_id. Perform the following activities:
1. Convert the list of entities into a CSV format that is machine loadable using the specification provided.
2. Provide the visit order for each entity as suggested by the text using the specification provided.

The list of entities is provided in the following pipe-delimited format:
entity_id | entity_type | entity_name | entity_subtype
Output the exact same information in a CSV format beginning and ending with <nodes> tags. The file will have this header 
followed by a row for each entity provided:
<nodes>
:ID, :LABEL, name:String, subtype:String, neighborhood:String, is_start:Bool
</nodes>
- The :ID field will contain the entity_id prefixed with the neighborhood_id field and an underscore
- The :LABEL field will be the entity_type value
- The name:String field will be the entity_name
- The subtype:String field will be the entity_subtype
- The is_start:Bool field will be true if this is suggested as the first stop in the neighborhood. Otherwise it will be false.

The visit order will be stored in a CSV file surrounded by <edges> tags as shown below:
<edges>
:ID, :START_ID, :END_ID, :TYPE
</edges>

- The order of visiting each entity is suggested by the text.
- If the text suggests the user can choose what to visit next, add a new line for each choice having the same :START_ID
- The first line is the header exactly as shown in the example.
- Every entity :ID should appear on at least one line in the file in the :START_ID column. If the next suggested entity is not in the neighborhood, 
list the :START_ID of the current entity, the neighborhood_id in the :END_ID column, and EXIT_NEIGHBORHOOD as the :TYPE field.
- :ID is a RFC 4122 compliant GUID
- :START_ID is the current entity :ID exactly as shown in <nodes>
- :END_ID is the next entity :ID exactly as shown in <nodes> unless it is the neighborhood_id as described in the scenario above.
- :TYPE is always HAS_NEXT_STOP unless it is EXIT_NEIGHBORHOOD as described in the scenario above.

Always ensure you have both the opening and closing tags nodes and edges

Here is an example of what the output may look like:
<nodes>
:ID,:LABEL,name:String,subtype:String,is_start:Bool
entity_id_1,LANDMARK,landmark 1,PARK,true
entity_id_2,BUSINESS,business 1,RESTAURANT,false
</nodes>
<edges>
:ID,:START_ID,:END_ID,:TYPE
guid_1,entity_id_1,entity_id_2,HAS_NEXT_STOP
guid_2,entity_id_2,neighborhood_id,EXIT_NEIGHBORHOOD
</edges>
"""

ENTITY_PROMPT_2["USER_PROMPT"] = """
Reference Text: {input_text}
Entity List:
{entity_list}
neighborhood_name: {neighborhood_name}
neighborhood_id: {neighborhood_id}
"""
    


for line in neighborhood_nodes[1:]:  # skip the header row
    fields = line.split(',')
    neighborhood_id = fields[0]
    neighborhood_name = fields[2]

    print(f"Starting neighborhood '{neighborhood_name}':")
    context["neighborhood_name"] = neighborhood_name
    context["neighborhood_id"] = neighborhood_id

    messages = []

    # The initial image has our reference text and is cached
    initial_message = {
        "role": "user",
        "content": [
            {
                "text": f"""
                            Neighborhood Name: {neighborhood_name}
                            Reference Text: {reference_text}
                        """
            },
            {
                "cachePoint": {
                    "type": "default"
                }
            }
        ]
    }

    messages.append(initial_message)
    should_continue_conversation = True
    iteration = 0
    MAX_ITERATIONS = 20
    
    while should_continue_conversation and iteration < MAX_ITERATIONS:
        iteration = iteration + 1
        print(f"conversation iteration {iteration} of up to {max_iterations}")

        llm_response = llm_conversation(
                messages, 
                context, 
                bedrock_client, 
                model_id, 
                model_metadata, 
                ENTITY_PROMPT_1, 
                cost_tracking=costs, 
                cache_system_prompt = True,
                tools = [tool_spec]
            )
        messages.append(llm_response["output"]["message"])

        tool_response = process_response(llm_response)
        should_continue_conversation = tool_response["continueConversation"]
        if should_continue_conversation:
            messages.append({"role": "user", "content": tool_response["response"]})
#        print(tool_response)

    # we've finished, so output the response for this neighborhood
    for item in tool_response["response"]:
        if "text" in item:
            print(f"===LLM Commentary==={NEWLINE}{item['text']}{NEWLINE}===End LLM Commentary===")
        else:
            print(item)
    
    entities = []
    for item in tool_response["response"]:
        if "text" in item:
            lines = item["text"].splitlines(keepends=True)
            in_entities_tag = False
    
            for line in lines:
                match line.strip(): 
                    case "<entities>": 
                        in_entities_tag = True
                    case "</entities>": 
                        in_entities_tag = False
                    case _:
                        if in_entities_tag:
                            entities.append(line.strip())
                        
    print(entities)
    
    context["entity_list"] = NEWLINE.join(entities)

    response_text = run_llm(reference_text,context,bedrock_client,model_id,model_metadata,ENTITY_PROMPT_2,cost_tracking=costs,print_prompts=False)

    lines = response_text.splitlines(keepends=True)
    
    in_nodes_tag = False
    in_edges_tag = False

    nodes_by_neighborhood[neighborhood_id] = []
    edges_by_neighborhood[neighborhood_id] = []
    nodes_store = nodes_by_neighborhood[neighborhood_id]
    edges_store = edges_by_neighborhood[neighborhood_id]
    for line in lines:
        match line.strip(): 
            case "<nodes>": 
                in_nodes_tag = True
                in_edges_tag = False
            case "</nodes>": 
                in_nodes_tag = False
            case "<edges>": 
                in_nodes_tag = False
                in_edges_tag = True
            case "</edges>": 
                in_edges_tag = False
            case _:
                if in_nodes_tag:
                    nodes_store.append(line.strip())
                elif in_edges_tag:
                    edges_store.append(line.strip())

for key in nodes_by_neighborhood:
    for line in nodes_by_neighborhood[key]:
        print(line)

for key in edges_by_neighborhood:
    for line in edges_by_neighborhood[key]:
        print(line)

print(costs.printVerbose())


Starting neighborhood ' Greenwich Village':
conversation iteration 1 of up to 20
===LLM Commentary===
I'll analyze the text and extract entities located in Greenwich Village. First, let me identify all the places mentioned in the Greenwich Village section and verify their locations.
===End LLM Commentary===
Calling Location_Tool for input {'name': 'St Luke in the Fields Garden', 'assumed_neighborhood': 'Greenwich Village'}
The top location only had a certainty score of 0.66 so returning no result.
conversation iteration 2 of up to 20
===LLM Commentary===
Let me try another entity:
===End LLM Commentary===
Calling Location_Tool for input {'name': 'Washington Square Park', 'assumed_neighborhood': 'Greenwich Village'}
Top match by the location service was label=Washington Square Park, 2 5th Ave, New York, NY 10011-0034, United States; neighborhood=Greenwich Village)
conversation iteration 3 of up to 20
Calling Location_Tool for input {'name': "Joe's Pizza", 'assumed_neighborhood': 'Greenw

You may notice some interesting observations as the LLM reasons through the results.  For example, it may detect Rockefeller Center as a neighborhood and then later reason it is actually a building in Midtown instead of a neighborhood once the location data returns.  In the commentary, you may see something like
<pre>Based on the tool responses, I now understand that the proper neighborhood name is "Midtown Center" and within it is the Rockefeller Center area.</pre>
Now we will write 5 CSV files to our local Notebook instance and later we will upload them to an S3 bucket created for us. One relationship we are adding that wasn't explicitly created by the LLM is one between each entity and the neighborhood they are located in.  

In [41]:
import uuid
entity_to_neighborhood_header = [":ID",":START_ID",":END_ID",":TYPE"]
entity_to_neighborhood_edges = []

with open('neighborhood_nodes.csv','w') as f:
    for line in neighborhood_nodes:
        f.write(f"{line}{NEWLINE}")
        
with open('neighborhood_edges.csv','w') as f:
    for line in neighborhood_edges:
        f.write(f"{line}{NEWLINE}")
        
with open('entity_nodes.csv','w') as f:
    f.write(f"{nodes_by_neighborhood[next(iter(nodes_by_neighborhood))][0]}{NEWLINE}")  # first print the header
    for key in nodes_by_neighborhood:
        for line in nodes_by_neighborhood[key][1:]:
            entity_to_neighborhood_edges.append(
                {
                    ":ID": uuid.uuid4(),
                    ":START_ID": line.split(",")[0],
                    ":END_ID": key,
                    ":TYPE": "IS_IN_NEIGHBORHOOD"
                }
            )
            f.write(f"{line}{NEWLINE}") 

with open('entity_edges.csv','w') as f:
    f.write(f"{edges_by_neighborhood[next(iter(edges_by_neighborhood))][0]}{NEWLINE}")  # first print the header
    for key in edges_by_neighborhood:
        for line in edges_by_neighborhood[key][1:]:  # skip the header in each subsection
            f.write(f"{line}{NEWLINE}")

with open('entity2neighborhood_edges.csv','w') as f:
    writer = csv.DictWriter(f, fieldnames=entity_to_neighborhood_header)
    writer.writeheader()
    writer.writerows(entity_to_neighborhood_edges)

Here we are uploading the files to an S3 bucket created specifically for this account.

In [42]:
import os
s3_client = boto3.client('s3')
bucket = os.environ['S3_WORKING_BUCKET']

files = ["neighborhood_nodes.csv","neighborhood_edges.csv","entity_nodes.csv","entity_edges.csv","entity2neighborhood_edges.csv"]
for file in files:
    try:
        response = s3_client.upload_file(file, bucket, file)
    except ClientError as e:
        logging.error(e)

loader_arn = json.loads(config)["load_from_s3_arn"]

Now we are going to run the Neptune Bulk Loader magic to load these files into Neptune.  Execute the cell and then just click the "Submit" button. All the properties you need will be pre-populated for you.

In [43]:
%load --source s3://{bucket}/ -l $loader_arn -f opencypher -p OVERSUBSCRIBE -r $AWS_REGION --no-fail-on-error -m AUTO --store-to loader_output

Button(description='Submit', style=ButtonStyle())

Output()

<div class="alert alert-block alert-warning">It is pretty likely that your status is LOAD_FAILED. Unfortunately, the issue likely arises in the LLM hallucinating IDs for the nodes when generating the edges. We'll talk about ways we can circumvent this at the end</div>

Here we are capturing the Load ID specific to our bulk loader job.

In [44]:
load_id = loader_output["payload"]["loadId"]

NameError: name 'loader_output' is not defined

Let's run another Neptune Notebook magic to see the details of our bulk load.

In [ ]:
%load_status {load_id} --errors --details

<div class="alert alert-block alert-warning">If your load failed, it is almost always in the entity_edges.csv file. The LLM often gets confused when generating the EXIT_NEIGHBORHOOD edges and adds characters to the ID for the neighborhood or uses "neighborhood_id" instead of substituting the actual files.  Again, we want to give you a real experience instead of just a scripted ideal one. You'll see lines in the errors section such as:
    <pre>{
          "errorCode": "FROM_OR_TO_VERTEX_ARE_MISSING",
          "errorMessage": "Either from vertex, 'neighborhood_top_of_the_rock_12', or to vertex, 'neighborhood_neighborhood_top_of_the_rock', is not present.",
          "fileName": "s3://cfn-deploy-s3workingbucket-uxyoloylt2td/entity_edges.csv",
          "recordNum": 0
        }</pre>
In this case, I can see 'neighborhood_neighborhood_top_of_the_rock' should actually be 'neighborhood_top_of_the_rock'. See below for next steps.
</div>

If your load failed, you have three options:
1. Go back and try to generate and load the files again. It may take several tries to get working files.
2. Track down the errors and fix them manually.  Then copy the files to S3 again and reload them.
3. Just run the cells below to load a set of files we manually fixed and move on.



Run these next 2 cells if your load failed and you chose option 3.  This will take a pre-cleansed copy of graph files and load them into your Neptune instance.  **If your load succeeded, then skip to the visualization step.**
<br>This next step will reset the storage on your Neptune cluster to clear out any partial data that may have loaded in the previous step.

<div class="alert alert-block alert-warning">Do NOT run these next three steps if your graph succeeded in loading</div>

In [45]:
%db_reset

Checkbox(value=False, description='I acknowledge that upon deletion the cluster data will no longer be availab…

Output()

In [46]:
files = ["neighborhood_nodes.csv","neighborhood_edges.csv","entity_nodes.csv","entity_edges.csv","entity2neighborhood_edges.csv"]
for file in files:
    try:
        response = s3_client.upload_file(f"backup_data/{file}", bucket, file)
    except ClientError as e:
        logging.error(e)

In [47]:
%load --source s3://{bucket}/ -l $loader_arn -f opencypher -p OVERSUBSCRIBE -r $AWS_REGION --no-fail-on-error -m AUTO --store-to loader_output

Button(description='Submit', style=ButtonStyle())

Output()

Next let's view a visualization of our notebook.  Run both cells. In the 2nd cell, you'll see a table of data and a tab labeled "Graph". Choose that tab to see a visualization of the graph we've extracted. This is a very expensive query, so We are going to limit the output to the first 2000 paths just to make sure we don't blow up the browser memory.

In [48]:
my_node_labels = '{"NEIGHBORHOOD":"name","LANDMARK":"name","MUSEUM":"name","BUSINESS":"name"}'

In [ ]:
%%oc -d $my_node_labels -l 20 -rel 20

MATCH p=(n:NEIGHBORHOOD)-[:NEXT_NEIGHBORHOOD*0..]->(:NEIGHBORHOOD)<-[:IS_IN_NEIGHBORHOOD]-()-[:HAS_NEXT_STOP*0..]->()
RETURN DISTINCT p
LIMIT 1000

Here we are going to heavily borrow from some code used in the [LangChain Neptune openCypher Q&A chain ](https://python.langchain.com/docs/integrations/graphs/amazon_neptune_open_cypher/) to allow us to generate openCypher queries from plain text.  Execute this cell to see the schema that is automatically generated from Neptune to inform the LLM about what our graph looks like so it can effectively generate a query.

In [50]:
schema_summary = neptune_client.get_propertygraph_summary()
nodeLabels = schema_summary["payload"]["graphSummary"]["nodeLabels"]
edgeLabels = schema_summary["payload"]["graphSummary"]["edgeLabels"]

types = {
    "str": "STRING",
    "float": "DOUBLE",
    "int": "INTEGER",
    "list": "LIST",
    "dict": "MAP",
    "bool": "BOOLEAN",
}

def _get_node_properties(n_labels, types):
    node_properties_query = """
    MATCH (a:`{n_label}`)
    RETURN properties(a) AS props
    LIMIT 100
    """
    node_properties = []
    for label in n_labels:
        q = node_properties_query.format(n_label=label)
        data = {"label": label, "properties": neptune_client.execute_open_cypher_query(openCypherQuery=q)["results"]}
        s = set({})
        for p in data["properties"]:
            for k, v in p["props"].items():
                s.add((k, types[type(v).__name__]))

        np = {
            "properties": [{"property": k, "type": v} for k, v in s],
            "labels": label,
        }
        node_properties.append(np)

    return node_properties
    
def _get_edge_properties(e_labels, types):
    edge_properties_query = """
    MATCH ()-[e:`{e_label}`]->()
    RETURN properties(e) AS props
    LIMIT 100
    """
    edge_properties = []
    for label in e_labels:
        q = edge_properties_query.format(e_label=label)
        data = {"label": label, "properties": neptune_client.execute_open_cypher_query(openCypherQuery=q)["results"]}
        s = set({})
        for p in data["properties"]:
            for k, v in p["props"].items():
                s.add((k, types[type(v).__name__]))

        ep = {
            "type": label,
            "properties": [{"property": k, "type": v} for k, v in s],
        }
        edge_properties.append(ep)

    return edge_properties
    
def _get_triples(e_labels):
    triple_query = """
    MATCH (a)-[e:`{e_label}`]->(b)
    WITH a,e,b LIMIT 3000
    RETURN DISTINCT labels(a) AS from, type(e) AS edge, labels(b) AS to
    LIMIT 10
    """

    triple_template = "(:`{a}`)-[:`{e}`]->(:`{b}`)"
    triple_schema = []
    for label in e_labels:
        q = triple_query.format(e_label=label)
        data = neptune_client.execute_open_cypher_query(openCypherQuery=q)["results"]
        for d in data:
            triple = triple_template.format(
                a=d["from"][0], e=d["edge"], b=d["to"][0]
            )
            triple_schema.append(triple)

    return triple_schema
    
node_properties = _get_node_properties(nodeLabels, types)
edge_properties = _get_edge_properties(edgeLabels, types)
triple_schema = _get_triples(edgeLabels)
schema = f"""
        Node properties are the following:
        {node_properties}
        Relationship properties are the following:
        {edge_properties}
        The relationships are the following:
        {triple_schema}
        """

print(schema)


        Node properties are the following:
        [{'properties': [{'property': 'is_start', 'type': 'BOOLEAN'}, {'property': 'subtype', 'type': 'STRING'}, {'property': 'name', 'type': 'STRING'}, {'property': 'neighborhood', 'type': 'STRING'}], 'labels': 'LANDMARK'}, {'properties': [{'property': 'is_start', 'type': 'BOOLEAN'}, {'property': 'name', 'type': 'STRING'}], 'labels': 'NEIGHBORHOOD'}, {'properties': [{'property': 'is_start', 'type': 'BOOLEAN'}, {'property': 'subtype', 'type': 'STRING'}, {'property': 'name', 'type': 'STRING'}, {'property': 'neighborhood', 'type': 'STRING'}], 'labels': 'BUSINESS'}, {'properties': [{'property': 'is_start', 'type': 'BOOLEAN'}, {'property': 'subtype', 'type': 'STRING'}, {'property': 'name', 'type': 'STRING'}, {'property': 'neighborhood', 'type': 'STRING'}], 'labels': 'MUSEUM'}]
        Relationship properties are the following:
        [{'type': 'NEXT_NEIGHBORHOOD', 'properties': []}, {'type': 'IS_IN_NEIGHBORHOOD', 'properties': []}, {'type': 'EXI

In [51]:
costs = CostTracking(model_id)

PROMPT = {}
PROMPT["SYSTEM_PROMPT"] = """
    Task:Generate Cypher statement to query a graph database.
    Instructions:
    Use only the provided relationship types and properties in the schema.
    Do not use any other relationship types or properties that are not provided.
    Schema:
    {schema}
    Note: Do not include any explanations or apologies in your responses.
    Do not respond to any questions that might ask anything else than for you to construct a Cypher statement.
    Do not include any text except the generated Cypher statement.
    """

PROMPT["USER_PROMPT"] = """
    The question is:
    {question}
"""

context["max_tokens"] = 8192
context["schema"] = schema
context["question"] = "I am currently in Greenwich Village. Is there a recommended restaurant nearby to eat at?"

response_text = run_llm("",context,bedrock_client,model_id,model_metadata,PROMPT,cost_tracking=costs)

print(response_text)

MATCH (n:NEIGHBORHOOD {name: "Greenwich Village"})
MATCH (b:BUSINESS {subtype: "restaurant"})-[:IS_IN_NEIGHBORHOOD]->(n)
RETURN b.name AS Restaurant
LIMIT 5


You can see that we have a query generated.  Now let's try to run it against our graph.

In [52]:
print(neptune_client.execute_open_cypher_query(openCypherQuery=response_text)["results"])

[]


It is pretty likely that you got no results.  The LLM likely correctly identified that it needed to filter for "restaurant" in the subquery filter, but it probably chose to use "restaurant" or "Restaurant". Unfortunately our model used "RESTAURANT".  OK, let's add an additional instruction to the prompt
<pre>The subtype field value is always upper-case.</pre>
So let's try this again now.

In [53]:
costs = CostTracking(model_id)

PROMPT = {}
PROMPT["SYSTEM_PROMPT"] = """
    Task:Generate Cypher statement to query a graph database.
    Instructions:
    Use only the provided relationship types and properties in the schema.
    Do not use any other relationship types or properties that are not provided.
    Schema:
    {schema}
    Note: Do not include any explanations or apologies in your responses.
    Do not respond to any questions that might ask anything else than for you to construct a Cypher statement.
    Do not include any text except the generated Cypher statement.
    The subtype field value is always upper-case.
    """

PROMPT["USER_PROMPT"] = """
    The question is:
    {question}
"""

context["max_tokens"] = 8192
context["schema"] = schema
context["question"] = "I am currently in Greenwich Village. Is there a recommended restaurant nearby to eat at?"

response_text = run_llm("",context,bedrock_client,model_id,model_metadata,PROMPT,cost_tracking=costs)

print(response_text)

query_results = neptune_client.execute_open_cypher_query(openCypherQuery=response_text)["results"]
print(query_results)

MATCH (n:NEIGHBORHOOD {name: 'Greenwich Village'})
MATCH (b:BUSINESS {subtype: 'RESTAURANT'})-[:IS_IN_NEIGHBORHOOD]->(n)
RETURN b.name AS Restaurant
[{'Restaurant': 'Stonewall Inn'}, {'Restaurant': "Joe's Pizza"}]


It is fairly likely that you got an answer to your question now! Notice that we didn't pass in the original full text of the walking tour here, the LLM is answering purely based on our graph.  If you want a natural language converstation for this, we can even incorporate this into a second prompt.

In [54]:
PROMPT = {}
PROMPT["SYSTEM_PROMPT"] = """
    You are an assistant that helps to form nice and human understandable answers based 
    on the provided information from a question and the graph query results. Do not add any other information that wasn't 
    present in the query result, and use very concise style in interpreting results!
"""

PROMPT["USER_PROMPT"] = """
    Question: {question}
    Graph Query Results: {query_results}
"""

context["max_tokens"] = 8192
context["query_results"] = query_results

response_text = run_llm("",context,bedrock_client,model_id,model_metadata,PROMPT,cost_tracking=costs)

print(response_text)

Based on the query results, there are two recommended restaurants nearby in Greenwich Village: Stonewall Inn and Joe's Pizza.


Could we have made this an even nicer experience for the user?

One way, we could have stored the full address obtained from the Location_Tool used earlier into our graph and then shown the user the exact location of the restaurant. 

Any other ideas?

## Extra Credit

We have a set of questions below along with the prompts we used for query generation and answer summarization. Can you adjust the prompts in a way that the LLM will be able to answer all 3 questions?

Hint: It may not be possible depending on how the LLM generated the graph.  If it is not possible, look at how we generated the graph and ideate how you might have better guided the model generation to be able to answer the question.  Is there a property or relationship type you could have asked the LLM to add?  A better defined ontology?

In [55]:
costs = CostTracking(model_id)

questions = [
    "I am currently in Midtown. What neighborhood should I go to next and what should I see there?",
    "Where does the walking tour end?",
    "I'm currently at St. Patrick's Cathedral and I'm hungry and I want to shop within several stops of there.  Where should I go?"
]

QUERY_GENERATION_PROMPT = {}
QUERY_GENERATION_PROMPT["SYSTEM_PROMPT"] = """
    Task:Generate Cypher statement to query a graph database.
    Instructions:
    Use only the provided relationship types and properties in the schema.
    Do not use any other relationship types or properties that are not provided.
    Schema:
    {schema}
    Note: Do not include any explanations or apologies in your responses.
    Do not respond to any questions that might ask anything else than for you to construct a Cypher statement.
    Do not include any text except the generated Cypher statement.
    The subtype field value is always upper-case.
    """

QUERY_GENERATION_PROMPT["USER_PROMPT"] = """
    The question is:
    {question}
"""

RESULT_SUMMARY_PROMPT = {}
RESULT_SUMMARY_PROMPT["SYSTEM_PROMPT"] = """
    You are an assistant that helps to form nice and human understandable answers based 
    on the provided information from a question and the graph query results. Do not add any other information that wasn't 
    present in the query result, and use very concise style in interpreting results!
"""

RESULT_SUMMARY_PROMPT["USER_PROMPT"] = """
    Question: {question}
    Graph Query Results: {query_results}
"""

for question in questions:

    context["question"] = question

    print(f"===QUESTION===\n{context['question']}\n===END QUESTION===")

    response_text = run_llm("",context,bedrock_client,model_id,model_metadata,QUERY_GENERATION_PROMPT,cost_tracking=costs)

    print(f"===QUERY===\n{response_text}\n===END QUERY===")

    query_results = neptune_client.execute_open_cypher_query(openCypherQuery=response_text)["results"]
    print(f"===RESULTS===\n{query_results}\n===END RESULTS===")

    context["query_results"] = query_results

    response_text = run_llm("",context,bedrock_client,model_id,model_metadata,RESULT_SUMMARY_PROMPT,cost_tracking=costs)

    print(f"===ANSWER===\n{response_text}\n===END ANSWER===")
    
print(costs.printVerbose())
    

===QUESTION===
I am currently in Midtown. What neighborhood should I go to next and what should I see there?
===END QUESTION===
===QUERY===
MATCH (current:NEIGHBORHOOD {name: 'Midtown'})
MATCH (current)-[:NEXT_NEIGHBORHOOD]->(next:NEIGHBORHOOD)
MATCH (poi)-[:IS_IN_NEIGHBORHOOD]->(next)
WHERE poi:LANDMARK OR poi:BUSINESS OR poi:MUSEUM
RETURN next.name AS NextNeighborhood, COLLECT(DISTINCT poi.name) AS PlacesToVisit
LIMIT 1
===END QUERY===
===RESULTS===
[]
===END RESULTS===
===ANSWER===
Based on the graph query results, there is no information available to recommend neighborhoods near Midtown or attractions to visit. I don't have data about nearby neighborhoods or attractions to suggest where you should go next.
===END ANSWER===
===QUESTION===
Where does the walking tour end?
===END QUESTION===
===QUERY===
MATCH (n) WHERE n.is_start = true
MATCH path = (n)-[:HAS_NEXT_STOP*]->(end)
WHERE NOT EXISTS((end)-[:HAS_NEXT_STOP]->())
RETURN end.name, end.neighborhood, labels(end) as type
===END Q